# esa-lookup -- TO Number & Notification Number processes

Fills the GTF SS Database workbook from SAP (transaction **ZTBV**, plant
**ESA1**). Same steps, same message boxes, and the same results as the
original per-step notebook -- but each step runs as **one SAP query + one
bulk Excel write**, so a full sheet takes seconds instead of minutes, the
multi-select filter is cleared of leftovers before every paste, and the
workbook is written **once, at the end** (never left half-filled: if a step
fails, the completed steps are written and the rest of the columns are left
exactly as they were).

## How to use -- 3 actions

1. Log into SAP GUI and open the workbook in Excel (or know its path).
2. Run the **Utility class for data pulling** cell first. Nothing to
   read or change in it, and running it again is always harmless.
3. Run YOUR process cell: **TO Number Process** or **Notification Number
   Process**. A message box reports each step: **OK** = continue,
   **Cancel** = stop with the workbook untouched.

On a combined sheet run the **TO process FIRST**, then the Notification
process -- the second pass fills the notification-only rows (usually most
of the sheet).

**Cell -> Run All works too**, and runs exactly that order: the utility
class, TO
process, Notification process. You browse to the workbook once; the second
process re-uses it automatically. If a process fails or you press Cancel,
the cells after it are stopped. The Diagnose cell at the bottom never runs
unless you uncomment it.

Every run also writes a log file to `%LOCALAPPDATA%\esa-lookup\logs\`
(last 20 kept). Attach it whenever reporting a problem.


## Utility class for data pulling -- always run this cell first

The data-pulling utilities both processes use: Excel read/write, SAP GUI
scripting, and the step runner. **Nothing in here needs reading or
editing.** Just run it, then go to your process below. Running it again at
any time is harmless. It is auto-generated from the app's source files by
`build_notebook.py`; to change behavior, change those files and regenerate.
Do not hand-edit this notebook.


In [ ]:
from __future__ import annotations

import contextlib
import difflib
import io
import os
import re
import sys
import tempfile
import threading
import time
import traceback
from dataclasses import dataclass, field
from decimal import Decimal, InvalidOperation
from typing import Callable

import pandas as pd
import pythoncom
import win32com.client
from win32com.client import constants  # noqa: F401 (loaded lazily by pywin32)


XL_UP = -4162  # Excel: xlUp
XL_CALC_MANUAL = -4135
XL_CALC_AUTOMATIC = -4105


class ExcelError(RuntimeError):
    pass


@dataclass
class ExcelCtx:
    app: object
    book: object
    sheet: object


def excel_attach(path: str) -> ExcelCtx:
    """Return an ExcelCtx for `path`. If Excel already has the file open,
    reuse it; otherwise start Excel and open it read/write.
    """
    if not os.path.exists(path):
        raise ExcelError(f"Excel file not found:\n{path}")

    abs_path = os.path.abspath(path).lower()

    app = None
    try:
        app = win32com.client.GetActiveObject("Excel.Application")
    except Exception:
        app = win32com.client.Dispatch("Excel.Application")
        app.Visible = True

    # Fix I: OneDrive/SharePoint-hosted workbooks report FullName as an
    # https:// URL, not a local filesystem path. os.path.abspath on a URL
    # produces gibberish and never matches -- so we then Open() a second
    # (read-only) copy and abort. Match by URL-tail filename in that case.
    target_basename = os.path.basename(path).lower()
    book = None
    for i in range(1, app.Workbooks.Count + 1):
        b = app.Workbooks(i)
        try:
            fn = str(b.FullName)
        except Exception:
            continue
        fn_lower = fn.lower()
        if fn_lower.startswith(("http:", "https:")):
            tail = fn.rsplit("/", 1)[-1].lower()
            if tail == target_basename:
                book = b
                break
        else:
            try:
                if os.path.abspath(fn).lower() == abs_path:
                    book = b
                    break
            except Exception:
                continue

    # Fix 7: always pass an absolute path -- Excel resolves relative paths
    # against its own cwd, which is usually not what the caller wants.
    if book is None:
        book = app.Workbooks.Open(os.path.abspath(path), 0, False)

    if book.ReadOnly:
        raise ExcelError(
            "Workbook is open Read-Only. Close it in Excel or check "
            "OneDrive/SharePoint permissions, then retry.\n" + path
        )

    return ExcelCtx(app=app, book=book, sheet=book.Worksheets(1))


def last_row_in_column(sheet, col_index: int) -> int:
    return int(sheet.Cells(sheet.Rows.Count, col_index).End(XL_UP).Row)


def read_range_2d(sheet, range_str: str) -> list[list]:
    """Read a range in one COM call; normalize to a list-of-lists.

    Single-cell ranges come back as a scalar; single-row/single-column
    ranges come back as a 1-tuple of tuples in some cases -- normalize both.
    """
    raw = sheet.Range(range_str).Value
    if raw is None:
        return []
    if not isinstance(raw, tuple):
        return [[raw]]
    # Range with a single row: raw is a tuple of scalars
    if raw and not isinstance(raw[0], tuple):
        return [list(raw)]
    return [list(r) for r in raw]


def write_range_2d(sheet, range_str: str, values_2d: list[list]) -> None:
    """Write a 2D python list to a range in one COM call.

    Excel's COM interface accepts a tuple-of-tuples on the RHS of Range.Value.
    Fix J: pre-check for merged cells so a mid-write failure can never leave
    the sheet in a half-updated state.
    """
    if not values_2d:
        return
    rng = sheet.Range(range_str)
    try:
        merged = bool(rng.MergeCells)
    except Exception:
        # `MergeCells` returns None (not a bool) for a mixed-merge range;
        # that itself signals merged content is present.
        merged = True
    if merged:
        raise ExcelError(
            f"Cannot write to range {range_str}: it contains merged cells. "
            f"Unmerge those columns in Excel and re-run. (SAP lookup output "
            f"columns must be plain, unmerged cells.)"
        )
    payload = tuple(tuple(row) for row in values_2d)
    rng.Value = payload


def clear_range(sheet, range_str: str) -> None:
    sheet.Range(range_str).ClearContents()


def set_column_format_text(sheet, columns: str) -> None:
    """columns like 'A:E' or 'M:M'."""
    sheet.Range(f"{columns}").NumberFormat = "@"


def autofit(sheet, columns: str) -> None:
    sheet.Columns(columns).AutoFit()


@contextlib.contextmanager
def bulk_write(app):
    """Disable Excel screen updating and calc while writing large blocks."""
    prev_updating = app.ScreenUpdating
    prev_events = app.EnableEvents
    prev_calc = app.Calculation
    try:
        app.ScreenUpdating = False
        app.EnableEvents = False
        app.Calculation = XL_CALC_MANUAL
        yield
    finally:
        app.Calculation = prev_calc
        app.EnableEvents = prev_events
        app.ScreenUpdating = prev_updating


def stage_values_on_clipboard(app, values: list[str]) -> object:
    """Create a scratch workbook, dump `values` into Column A as text,
    Copy() them onto the OS clipboard, and return the scratch workbook so
    the caller can close it after SAP has consumed the paste.
    """
    scratch = app.Workbooks.Add()
    ws = scratch.Worksheets(1)
    ws.Columns("A").NumberFormat = "@"
    # Fix 1: one bulk COM assignment instead of N per-cell writes. On a
    # 5000-key input this is the difference between ~30 s and <1 s.
    if values:
        payload = tuple(("" if v is None else str(v),) for v in values)
        ws.Range(f"A1:A{len(values)}").Value = payload
        ws.Range(f"A1:A{len(values)}").Copy()
    return scratch


def close_scratch(scratch) -> None:
    try:
        scratch.Close(SaveChanges=False)
    except Exception:
        pass


def excel_save(book) -> None:
    """Save the workbook. Fix L: raise ExcelError on failure instead of
    silently swallowing it, so the pipeline can log a warning and the user
    knows to save manually rather than assuming success.
    """
    try:
        book.Save()
    except Exception as e:
        raise ExcelError(
            f"Excel refused to save the workbook: {e}. Data is written but "
            f"unsaved -- press Ctrl+S in Excel to persist it."
        ) from e


PLANT = "ESA1"
TRANSACTION = "ZTBV"


class SapError(RuntimeError):
    pass


@dataclass
class SapSession:
    session: object

    def find(self, oid):
        return self.session.findById(oid)


def sap_attach() -> SapSession:
    """Attach to the first running SAP GUI session. Raise SapError if none."""
    try:
        sap_gui = win32com.client.GetObject("SAPGUI")
    except Exception as e:
        raise SapError(
            "Cannot reach SAP GUI. Log into SAP GUI first and confirm "
            "'Enable scripting' is on (Options -> Accessibility & Scripting)."
        ) from e
    try:
        app = sap_gui.GetScriptingEngine
        conn = app.Children(0)
        sess = conn.Children(0)
    except Exception as e:
        raise SapError(
            "SAP GUI is running but no active session was found. Open a "
            "connection and log in, then retry."
        ) from e
    return SapSession(sess)


def close_lingering_modals(s: SapSession, log=None) -> int:
    """Fix K: close any wnd[1..N] popups left over from a previous run.
    Typing an okcd on wnd[0] while a modal is open is silently ignored,
    which then causes the next few button-presses to hit the WRONG screen.
    Returns the number of modals actually closed (for diagnostic logging).
    """
    closed = 0
    for i in range(9, 0, -1):
        try:
            title = ""
            try:
                title = str(s.find(f"wnd[{i}]").Text or "")
            except Exception:
                pass
            s.find(f"wnd[{i}]").close()
            closed += 1
            if log:
                log(f"SAP: closed leftover modal wnd[{i}]" +
                    (f" (title: {title!r})" if title else ""))
        except Exception:
            pass
    return closed


def open_ztbv_table(s: SapSession, table: str, log=None) -> None:
    """Navigate to /nZTBV -> plant + table -> F8 (enter table view)."""
    if log:
        log(f"SAP: /n{TRANSACTION} -> {table} @ plant {PLANT}")
    n_closed = close_lingering_modals(s, log=log)
    if n_closed and log:
        log(f"SAP: {n_closed} lingering modal(s) closed before navigation")
    s.find("wnd[0]").maximize()
    s.find("wnd[0]/tbar[0]/okcd").Text = f"/n{TRANSACTION}"
    s.find("wnd[0]").sendVKey(0)
    time.sleep(0.3)
    s.find("wnd[0]/usr/txtD_WERKS").Text = PLANT
    s.find("wnd[0]/usr/ctxtD_TAB").Text = table
    s.find("wnd[0]/usr/ctxtD_TAB").SetFocus()
    s.find("wnd[0]/usr/ctxtD_TAB").caretPosition = len(table)
    s.find("wnd[0]/tbar[1]/btn[8]").press()  # F8 -> selection screen
    time.sleep(0.3)


_VALU_PUSH_RE = re.compile(r"btn%_(.+?)_%_APP_%-VALU_PUSH")
_SELOPT_INPUT_RE = re.compile(r"(S\d+)-(LOW|HIGH)$", re.I)

# The word SAP prints between a select-option's LOW and HIGH fields. It is
# NOT a field name, but it is the closest text to the left of the '=>'
# button, so it wins any "nearest label" rule. Logon-language dependent --
# the leftmost-text rule in `label_for` is the real defence; this list just
# makes a single-text row fail cleanly instead of reporting the separator.
_RANGE_SEPARATORS = frozenset({
    "to", "bis", "a", "à", "hasta", "até", "fino a", "tot", "till", "do",
    "~", "-", "..",
})


def _is_range_separator(text: str) -> bool:
    return text.strip().lower() in _RANGE_SEPARATORS


def _attr(c, name: str, default: str = "") -> str:
    try:
        v = getattr(c, name)
        return "" if v is None else str(v)
    except Exception:
        return default


def _int_attr(c, name: str, default: int = -1) -> int:
    try:
        return int(getattr(c, name))
    except Exception:
        return default


def _walk_controls(root, max_depth: int = 6) -> list:
    """Every control under `root`, depth-first. Individual failures are
    skipped -- a selection screen with one unreadable control should still
    yield the other 200."""
    out = []

    def walk(node, depth):
        if depth > max_depth:
            return
        try:
            children = node.Children
            count = int(children.Count)
        except Exception:
            return
        for i in range(count):
            try:
                c = children(i)
            except Exception:
                continue
            out.append(c)
            walk(c, depth + 1)

    walk(root, 0)
    return out


def screen_inventory(s, log=None) -> list[dict]:
    """Flat dump of every control on wnd[0]/usr, with both coordinate systems.

    This is the raw material for `describe_selection_screen` and for the
    Diagnose run -- when label pairing fails, the inventory is what lets a
    human map S<n> -> field by eye from the log file.
    """
    try:
        usr = s.find("wnd[0]/usr")
    except Exception as e:
        raise SapError(f"No selection screen on wnd[0]/usr: {e}") from e

    items = []
    for c in _walk_controls(usr):
        cid = _attr(c, "Id")
        items.append({
            "id": cid[cid.find("wnd[0]"):] if "wnd[0]" in cid else cid,
            "type": _attr(c, "Type"),
            "name": _attr(c, "Name"),
            "text": _attr(c, "Text").strip(),
            "tooltip": _attr(c, "Tooltip").strip(),
            # Character metric: exact per dynpro row/column. Only meaningful
            # for controls inside the user area -- which is all of these.
            "row": _int_attr(c, "CharTop"),
            "col": _int_attr(c, "CharLeft"),
            # Pixel metric: the fallback when CharTop/CharLeft read as 0 on
            # this SAP GUI build.
            "ptop": _int_attr(c, "Top"),
            "pleft": _int_attr(c, "Left"),
        })
    if log:
        log(f"SAP: selection screen carries {len(items)} control(s)")
    return items


def describe_selection_screen(s, log=None) -> list[dict]:
    """List every multi-value ('=>' arrow) filter on the current ZTBV
    selection screen, with the on-screen label next to each one.

    ZTBV names its select-options generically -- S3, S15, S29 -- so which
    slot is which field cannot be inferred from the id alone, and guessing
    would paste values into the wrong filter and return confidently wrong
    data. Run this once against a table to read the mapping off the screen
    instead of recording it by hand.

    Pairing runs in CHARACTER metric (CharTop/CharLeft), not pixels. A
    selection screen row reads [label] [LOW] [HIGH] [=> button], so the
    field name is the nearest label to the LEFT on the SAME dynpro row.
    Pixel `Top` was what the previous version used, and it does not agree
    between a label and a push button drawn on one row -- every filter came
    back unlabelled.

    Only GuiLabel controls are treated as labels. The select-option input
    fields on the same row are GuiTextField/GuiCTextField and hold retained
    FILTER VALUES, so accepting them would let a leftover TO number pose as
    a field name.

    Each filter's `tooltip` (read off its own -LOW field) is carried as an
    independent second source of the field's identity, used by
    `resolve_push_button` only when no label matched.

    Call it AFTER open_ztbv_table(), while the selection screen is up.
    Returns [{param, label, tooltip, push_id, low_field, row, col}].
    """
    items = screen_inventory(s, log=None)

    # CharTop/CharLeft are 0 on some SAP GUI builds for some control types.
    # Fall back to pixels wholesale rather than mixing the two metrics.
    use_char = any(it["row"] > 0 for it in items)
    row_of = (lambda it: it["row"]) if use_char else (lambda it: it["ptop"])
    col_of = (lambda it: it["col"]) if use_char else (lambda it: it["pleft"])
    # Row tolerance for the "near miss" pass: 1 dynpro row, or roughly one
    # row of pixels when we are stuck in pixel metric.
    row_slack = 1 if use_char else 12

    labels = [it for it in items if it["type"] == "GuiLabel" and it["text"]]
    if not labels:
        # No GuiLabel at all: widen to text controls that are NOT select-option
        # inputs, so a screen built entirely from output fields still resolves.
        # Noted in the log because these are a weaker signal than a real label.
        labels = [
            it for it in items
            if it["text"]
            and it["type"] in ("GuiTextField", "GuiCTextField")
            and not _SELOPT_INPUT_RE.search(it["name"] or "")
        ]
        if labels and log:
            log(f"SAP: no GuiLabel on this screen; falling back to "
                f"{len(labels)} non-input text control(s) as label sources")

    # Tooltip of each select-option's own -LOW field, keyed by S<n>.
    tooltip_by_param: dict[str, str] = {}
    low_field_by_param: dict[str, str] = {}
    for it in items:
        m = _SELOPT_INPUT_RE.search(it["name"] or "")
        if not m or m.group(2).upper() != "LOW":
            continue
        param = m.group(1).upper()
        low_field_by_param[param] = it["id"]
        if it["tooltip"]:
            tooltip_by_param[param] = it["tooltip"]

    def row_texts(btn) -> list[str]:
        """Every text on the button's row, left to right. Carried into the
        result so a mis-picked label can be diagnosed from one log."""
        brow, bcol = row_of(btn), col_of(btn)
        return [l["text"] for l in
                sorted((l for l in labels
                        if row_of(l) == brow and col_of(l) < bcol),
                       key=col_of)]

    # A select-options row reads:
    #     [field name]  [LOW]  "to"  [HIGH]  [=> button]
    # The RANGE SEPARATOR is therefore the nearest text to the left of the
    # button and never the field name -- a "nearest label" rule reported all
    # 39 filters on the ESA screen as 'to'. Two defences: separators are not
    # label candidates at all, and the field name is taken as the LEFTMOST
    # text on the row rather than the nearest.
    named = [l for l in labels if not _is_range_separator(l["text"])]

    def label_for(btn) -> str:
        brow, bcol = row_of(btn), col_of(btn)
        same_row = [l for l in named if row_of(l) == brow]
        left = [l for l in same_row if col_of(l) < bcol]
        if left:
            return min(left, key=col_of)["text"]
        if same_row:
            return min(same_row, key=col_of)["text"]
        near = [l for l in named if abs(row_of(l) - brow) <= row_slack]
        if near:
            return min(near, key=lambda l: (abs(row_of(l) - brow), -col_of(l)))["text"]
        return ""

    out: list[dict] = []
    for it in items:
        m = _VALU_PUSH_RE.search(it["id"])
        if not m:
            continue
        param = m.group(1).upper()
        label = label_for(it)
        tooltip = tooltip_by_param.get(param, "")
        texts = row_texts(it)
        out.append({
            "param": param,
            "label": label or "(no label found)",
            "tooltip": tooltip,
            "row_texts": texts,
            "push_id": it["id"],
            "low_field": low_field_by_param.get(
                param, f"wnd[0]/usr/ctxt{param}-LOW"),
            "row": row_of(it),
            "col": col_of(it),
        })
        if log:
            log(f"SAP: filter {param:<6} label={label or '(none)'!r}"
                + (f" tooltip={tooltip!r}" if tooltip else "")
                + (f" row={texts}" if len(texts) > 1 else ""))
    if not out:
        raise SapError(
            "No multi-value filter buttons found. Is the ZTBV selection "
            "screen actually displayed (call open_ztbv_table first)?")
    if log:
        named = sum(1 for f in out if f["label"] != "(no label found)")
        log(f"SAP: {len(out)} filter(s) found, {named} with a label, "
            f"{sum(1 for f in out if f['tooltip'])} with a tooltip "
            f"({'character' if use_char else 'pixel'} metric)")
    return out


def paste_multi_value_filter(
    s: SapSession, push_button_id: str, values: list[str], log=None
) -> None:
    """Click multi-select push button, upload clipboard, OK.

    Caller is responsible for having already staged `values` on the OS
    clipboard (typically via Excel: put values in Column A of a temp
    workbook then .Copy()). This mirrors the notebook's approach.
    """
    if log:
        log(f"SAP: pasting {len(values)} filter values via clipboard")
    s.find(push_button_id).press()
    time.sleep(0.3)
    # Item 2: clear values retained from a previous chunk/run first -- SAP
    # keeps the selection dialog's contents within a session, and the
    # clipboard upload can append rather than replace. Deleting everything
    # up front makes the paste deterministic. btn[16] = "Delete Entire
    # Selection" on the standard multi-select dialog toolbar.
    try:
        s.find("wnd[1]/tbar[0]/btn[16]").press()
        time.sleep(0.2)
    except Exception:
        pass  # button absent on this SAP GUI version -- old behavior
    # In the multi-value dialog: btn[24] = Upload from Clipboard, btn[8] = OK
    s.find("wnd[1]/tbar[0]/btn[24]").press()
    time.sleep(0.3)
    s.find("wnd[1]/tbar[0]/btn[8]").press()
    time.sleep(0.2)


def fill_multi_value_filter_from_file(
    s: SapSession, push_button_id: str, values: list[str], work_dir: str,
    log=None,
) -> None:
    """Item 3: feed the multi-select filter from a temp TEXT FILE instead of
    the OS clipboard.

    The clipboard is global state -- a user copying anything while a run is
    in flight silently replaces the filter values (wrong results, not a
    crash). The multi-select dialog's 'Import from Text File' button
    (btn[23]) reads from a file only we control.

    Requires 'Show native Microsoft Windows dialogs' = Off (README end-user
    setup) so the file-open dialog is the scriptable SAP one (same
    DY_PATH / DY_FILENAME layout as the ALV export save dialog). Raises
    SapError on any failure -- after closing the dialogs it opened -- so
    the caller can fall back to the clipboard path.
    """
    os.makedirs(work_dir, exist_ok=True)
    filename = "esa_lookup_filter.txt"
    path = os.path.join(work_dir, filename)
    # One value per line, CRLF. SAP keys are plain ASCII; anything else
    # fails fast here and the caller drops to the clipboard path.
    try:
        with open(path, "w", encoding="ascii", newline="") as f:
            f.write("\r\n".join(str(v) for v in values))
            f.write("\r\n")
    except (OSError, UnicodeEncodeError) as e:
        raise SapError(f"cannot write filter file {path}: {e}") from e

    if log:
        log(f"SAP: importing {len(values)} filter values from file")
    try:
        # The opening press is INSIDE the try on purpose: a raw com_error
        # here (stale control, screen not ready) must surface as SapError
        # so the caller can attempt the clipboard fallback, not kill the
        # run with an undiagnosed COM code.
        s.find(push_button_id).press()
        time.sleep(0.3)
        # Clear leftover values first (same rationale as the clipboard
        # path: the dialog retains contents within a session).
        try:
            s.find("wnd[1]/tbar[0]/btn[16]").press()  # Delete Entire Selection
            time.sleep(0.2)
        except Exception:
            pass
        s.find("wnd[1]/tbar[0]/btn[23]").press()      # Import from Text File
        time.sleep(0.4)
        s.find("wnd[2]/usr/ctxtDY_PATH").Text = work_dir
        s.find("wnd[2]/usr/ctxtDY_FILENAME").Text = filename
        s.find("wnd[2]/tbar[0]/btn[0]").press()       # Open / OK
        time.sleep(0.3)
        s.find("wnd[1]/tbar[0]/btn[8]").press()       # Copy -> selection screen
        time.sleep(0.2)
    except Exception as e:
        # Leave the screen usable for the clipboard fallback: close the
        # file dialog and the multi-select dialog if still open.
        for wid in ("wnd[2]", "wnd[1]"):
            try:
                s.find(wid).close()
            except Exception:
                pass
        raise SapError(
            f"file import into the multi-select dialog failed: "
            f"{type(e).__name__}: {e}"
        ) from e


def execute_query(s: SapSession, log=None) -> None:
    """Press F8 on the selection screen to run the query."""
    if log:
        log("SAP: executing query (F8)")
    s.find("wnd[0]/tbar[1]/btn[8]").press()
    time.sleep(0.5)


def read_statusbar(s: SapSession) -> tuple[str, str]:
    """Return (message_type, text) from the main window's status bar.

    message_type is one of '' / 'S' (success) / 'W' (warning) / 'E' (error)
    / 'A' (abort) / 'I' (info). Returns ('', '') when the bar is empty or
    unreadable -- callers must treat that as "no message", not success.
    """
    try:
        sbar = s.find("wnd[0]/sbar")
        return (str(sbar.MessageType or "").strip().upper(),
                str(sbar.Text or "").strip())
    except Exception:
        return "", ""


def screen_snapshot(s: SapSession) -> str:
    """One plain-text snapshot of where SAP currently stands: main window
    title, any open popup titles, and the status bar message.

    Used by the error popup. The operator standing at the machine sees the
    frozen SAP screen; the person debugging remotely sees only what we
    capture here -- so grab everything cheap and never raise.
    """
    parts = []
    try:
        parts.append(f"window:  {str(s.find('wnd[0]').Text or '').strip()}")
    except Exception:
        parts.append("window:  (unreadable -- SAP GUI may be gone)")
    for i in (1, 2):
        try:
            title = str(s.find(f"wnd[{i}]").Text or "").strip()
            parts.append(f"popup wnd[{i}]: {title or '(untitled)'}")
        except Exception:
            pass  # no popup at this level -- the common case
    msg_type, msg_text = read_statusbar(s)
    if msg_text:
        parts.append(f"status bar: [{msg_type or ' '}] {msg_text}")
    return "\n".join(parts)


def query_result_check(s: SapSession, log=None) -> int:
    """Item 2: after F8, confirm a result grid exists and return its row
    count (0 = query ran but matched nothing).

    Reads the status bar first: an 'E' (error) or 'A' (abort) message, or a
    missing result grid, raises SapError carrying SAP's own message -- so a
    bad query fails HERE with a precise reason instead of two calls later
    as a confusing export failure. Non-fatal messages are just logged.
    """
    msg_type, msg_text = read_statusbar(s)
    if msg_text and log:
        log(f"SAP status bar: [{msg_type or ' '}] {msg_text}")
    if msg_type in ("E", "A"):
        raise SapError(f"SAP reported an error after executing the query: {msg_text}")
    try:
        grid = s.find("wnd[0]/shellcont/shell")
        return int(grid.RowCount)
    except Exception as e:
        raise SapError(
            "No result grid appeared after executing the query"
            + (f" -- SAP said: {msg_text!r}" if msg_text else "")
            + ". Check the filter values, table name, and plant."
        ) from e


def export_alv_to_file(
    s: SapSession, target_dir: str, filename: str, log=None, timeout_s: int = 30
) -> str:
    """Export the current ALV grid to a spreadsheet file in `target_dir`.

    Tries the modern `&XXL` toolbar path first, then falls back to `&PC`
    (Save as Local File) with the spreadsheet format radio.

    Returns the full path to the exported file. Raises SapError on failure
    or timeout.
    """
    os.makedirs(target_dir, exist_ok=True)
    target_path = os.path.join(target_dir, filename)
    # Fix A: os.remove can silently fail (AV / lingering handle). Even if the
    # stale file survives, we only accept files whose mtime is newer than the
    # moment we triggered the export, so we can never return a prior step's
    # data as the current step's result.
    if os.path.exists(target_path):
        try:
            os.remove(target_path)
        except OSError:
            pass
    export_started_at = time.time()

    grid = s.find("wnd[0]/shellcont/shell")

    last_err: Exception | None = None
    for approach in ("XXL", "PC"):
        try:
            if approach == "XXL":
                if log:
                    log("SAP: exporting via &MB_EXPORT / &XXL")
                grid.pressToolbarContextButton("&MB_EXPORT")
                time.sleep(0.2)
                grid.selectContextMenuItem("&XXL")
            else:
                if log:
                    log("SAP: retrying export via &PC")
                grid.pressToolbarContextButton("&MB_EXPORT")
                time.sleep(0.2)
                grid.selectContextMenuItem("&PC")
                time.sleep(0.4)
                # Format picker -- select spreadsheet radio if present
                try:
                    s.find(
                        "wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/"
                        "radSPOPLI-SELFLAG[1,0]"
                    ).Select()
                except Exception:
                    pass
                try:
                    s.find("wnd[1]/tbar[0]/btn[0]").press()
                except Exception:
                    pass
            time.sleep(0.5)
            # Save-As dialog: DY_PATH + DY_FILENAME + Save
            s.find("wnd[1]/usr/ctxtDY_PATH").Text = target_dir
            s.find("wnd[1]/usr/ctxtDY_FILENAME").Text = filename
            # btn[11] = "Replace" / Save on standard SAP save-as dialog
            s.find("wnd[1]/tbar[0]/btn[11]").press()
            # Wait for file, refusing any pre-existing stale copy (Fix A).
            deadline = time.time() + timeout_s
            while time.time() < deadline:
                if os.path.exists(target_path) and os.path.getsize(target_path) > 0:
                    try:
                        fresh = os.path.getmtime(target_path) >= export_started_at
                    except OSError:
                        fresh = False
                    if fresh:
                        if log:
                            log(f"SAP: export saved -> {target_path}")
                        return target_path
                time.sleep(0.25)
            raise SapError(f"Export timed out (>{timeout_s}s) waiting for {target_path}")
        except Exception as e:
            last_err = e
            if log:
                log(f"SAP: {approach} export attempt failed: "
                    f"{type(e).__name__}: {e}")
            # Best-effort: close any modal that might be hanging around so
            # the fallback attempt (or a subsequent step) can navigate.
            try:
                s.find("wnd[1]/tbar[0]/btn[12]").press()  # Cancel
                if log:
                    log("SAP: cancelled leftover modal to prepare fallback")
            except Exception:
                pass
            time.sleep(0.3)
            continue

    raise SapError(
        "ALV export failed via both &XXL and &PC. The exact save-dialog "
        "field IDs may differ on this SAP GUI version -- record one export "
        "via the SAP GUI Script Recorder and adjust `export_alv_to_file`."
    ) from last_err


def _visible_row_count(grid) -> int:
    """Rows the ALV control currently holds in the frontend buffer."""
    try:
        n = int(grid.VisibleRowCount)
        if n > 0:
            return n
    except Exception:
        pass
    return 20


def _scroll_grid(grid, row_number: int) -> bool:
    try:
        grid.firstVisibleRow = row_number
        return True
    except Exception:
        pass
    try:
        grid.VerticalScrollbar.Position = row_number
        return True
    except Exception:
        pass
    return False


def _safe_cell(grid, row: int, column: str) -> str:
    """GetCellValue with the original's scroll-and-retry: a row can fall out
    of the frontend buffer between the scroll and the read.
    """
    for _ in range(3):
        try:
            return grid.GetCellValue(row, column)
        except Exception:
            _scroll_grid(grid, row)
            time.sleep(0.2)
    return ""


def read_alv_grid(
    s: SapSession,
    columns: list[str],
    log=None,
    stop=None,
) -> list[dict]:
    """Read `columns` straight off the ALV grid, by TECHNICAL field name.

    This is the original notebook's read path (GetCellValue + scroll), and
    it is why that version never had to care what the layout titles its
    columns: 'TANUM' is 'TANUM' whether the ALV displays it as 'TO Number'
    or 'Transfer Order'. No alias table, no ambiguity when a layout repeats
    a title, no dependence on the export file format.

    Only the requested columns are fetched, so the cost is
    rows x len(columns) -- not rows x every field in the layout.

    Raises SapError if a name is not in the grid (wrong technical name, or
    the field is absent from the displayed variant) so the caller can fall
    back to the file export.
    """
    grid = s.find("wnd[0]/shellcont/shell")
    try:
        row_count = int(grid.RowCount)
    except Exception as e:
        raise SapError(f"ALV grid exposes no RowCount: {e}") from e
    if row_count == 0:
        return []

    # Fail fast on a bad column name: probe row 0 once per column before
    # committing to a full scroll-and-read pass.
    for c in columns:
        try:
            grid.GetCellValue(0, c)
        except Exception as e:
            raise SapError(
                f"ALV grid has no column {c!r} (technical name). Either the "
                f"field is missing from the displayed layout variant, or the "
                f"name differs on this system. Original error: {e}"
            ) from e

    visible = _visible_row_count(grid)
    if log:
        log(f"SAP: reading {row_count} row(s) x {len(columns)} column(s) "
            f"from the ALV grid by technical name")

    out: list[dict] = []
    for start in range(0, row_count, visible):
        if stop is not None and stop.is_set():
            break
        _scroll_grid(grid, start)
        time.sleep(0.05)
        for r in range(start, min(start + visible, row_count)):
            out.append({c: _safe_cell(grid, r, c) for c in columns})
    return out


# Multi-value push button IDs on the ZTBV selection screen per (table, field).
# These match the notebook. If a site uses a customized ZTBV layout the S<n>
# indices may shift -- record once with the Script Recorder and update here.
PUSH_BUTTONS = {
    ("LTAP", "TO_NUMBER"): "wnd[0]/usr/btn%_S3_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_CRNT", "RSNUM"): "wnd[0]/usr/btn%_S15_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_CRNT", "QMNUM"): "wnd[0]/usr/btn%_S29_%_APP_%-VALU_PUSH",
    # Read off the ESA ZTBV 'Table Display' screen (2026-08). That screen
    # carries 39 select-options, S2..S40, one per field in screen order, so
    # the slot is the field's position + 1. Two independent checks against
    # the mappings recorded from the original working notebook confirm the
    # offset: 'Number of Reservation/Depend' is field 14 -> S15 (= RSNUM
    # above) and 'Notification No' is field 28 -> S29 (= QMNUM above).
    # 'Transfer Order Number' is field 18 -> S19.
    ("Z50CFG_ENG_CRNT", "TO_NUMBER"): "wnd[0]/usr/btn%_S19_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_VALD", "OBJNR"): "wnd[0]/usr/btn%_S2_%_APP_%-VALU_PUSH",
}

# On-screen label fragments per logical field, all lowercase. Used by
# resolve_push_button to find a filter slot that PUSH_BUTTONS does not list
# yet -- ZTBV's S<n> parameter names are generic, but the label printed next
# to each filter names the field, so match on that.
FIELD_LABEL_SYNONYMS = {
    "TO_NUMBER": ["transfer order", "to number", "to no"],
    "RSNUM": ["reservation"],
    "QMNUM": ["notification", "notifctn"],
    "OBJNR": ["object"],
}

# Label fragments that RULE a filter OUT for a field, applied only to break a
# tie. Every field we look up is paired on the ESA screen with a neighbour
# that matches the same synonym:
#
#   'Transfer Order Number'          vs 'Transfer order item'
#   'Number of Reservation/Depend'   vs 'Item Number of Reservation/D'
#   'Notification No'                vs 'Notification Type'
#
# Each of these wants the NUMBER, never the item or the type. Without this,
# resolution correctly refuses as ambiguous -- safe, but it stops the run.
# Same failure mode as RSPOS once binding to the TO item (3d8644f).
FIELD_LABEL_EXCLUSIONS = {
    "TO_NUMBER": ["item", "itm", "pos"],
    "RSNUM": ["item", "itm", "pos"],
    "QMNUM": ["type", "typ"],
    "OBJNR": ["type", "typ"],
}


def push_button_id(param: str) -> str:
    """S7 -> the full multi-value push button id for that select-option."""
    return f"wnd[0]/usr/btn%_{param.upper()}_%_APP_%-VALU_PUSH"


def env_override_var(table: str, field: str) -> str:
    return "ESA_LOOKUP_PUSH_" + re.sub(
        r"[^A-Z0-9]+", "_", f"{table}_{field}".upper())


def _env_override(table: str, field: str) -> str | None:
    """Read a (table, field) -> filter mapping out of the environment.

    Lets whoever is standing at the customer's machine correct a mapping
    without a rebuild-and-redistribute cycle:

        set ESA_LOOKUP_PUSH_Z50CFG_ENG_CRNT_TO_NUMBER=S7

    Accepts either the bare select-option name ("S7") or a full control id.
    """
    raw = (os.environ.get(env_override_var(table, field)) or "").strip()
    if not raw:
        return None
    if re.fullmatch(r"S\d+", raw, re.I):
        return push_button_id(raw)
    return raw


def resolve_push_button(s, table: str, field: str, log=None) -> str:
    """Return the multi-value push button id for (table, field).

    Resolution order:
      1. An ESA_LOOKUP_PUSH_<TABLE>_<FIELD> environment override.
      2. PUSH_BUTTONS, the recorded mappings.
      3. The CURRENTLY DISPLAYED selection screen: the filter whose on-screen
         label matches the field's synonyms, else -- only if no label matched
         at all -- the filter whose tooltip does.

    Exactly one candidate must match. Zero or several raise SapError listing
    every filter on the screen, because pasting keys into the wrong filter
    would return plausible-looking wrong data -- the one failure mode worse
    than stopping.

    A resolved id is cached into PUSH_BUTTONS for the rest of the process.
    Must be called AFTER open_ztbv_table() so the screen is displayed.
    """
    key = (table, field)
    override = _env_override(table, field)
    if override:
        PUSH_BUTTONS[key] = override
        if log:
            log(f"SAP: {field} on {table} -> {override} "
                f"(from {env_override_var(table, field)})")
        return override
    if key in PUSH_BUTTONS:
        return PUSH_BUTTONS[key]

    synonyms = FIELD_LABEL_SYNONYMS.get(field)
    if not synonyms:
        raise SapError(
            f"No push button known for {key} and no label synonyms defined "
            f"for {field!r} -- add it to FIELD_LABEL_SYNONYMS or "
            f"PUSH_BUTTONS in ")

    filters = describe_selection_screen(s, log=log)
    exclusions = FIELD_LABEL_EXCLUSIONS.get(field, [])

    def pick(attr: str) -> list[dict]:
        hits = [f for f in filters
                if any(syn in f[attr].lower() for syn in synonyms)]
        # Only break a tie with the exclusions -- never let them turn a
        # single clean hit into no hit at all.
        if len(hits) > 1 and exclusions:
            narrowed = [f for f in hits
                        if not any(x in f[attr].lower() for x in exclusions)]
            if len(narrowed) == 1:
                return narrowed
        return hits

    hits = pick("label")
    matched_on = "label"
    if not hits:
        # Tooltips are only consulted when no label matched -- consulting both
        # at once would turn a clean single label hit into an ambiguity error.
        hits = pick("tooltip")
        matched_on = "tooltip"
    if len(hits) != 1:
        # Print every text on each row, not just the one picked as the label:
        # if the pick is wrong, the right answer is already in this listing.
        listing = "\n".join(
            f"  {f['param']:<6} label={f['label']!r}"
            + (f" tooltip={f['tooltip']!r}" if f["tooltip"] else "")
            + (f" row={f['row_texts']}" if len(f.get("row_texts", [])) > 1
               else "")
            for f in filters)
        raise SapError(
            f"Could not resolve the {field} filter on {table}: "
            f"{len(hits)} candidate(s) matched {synonyms}. Filters on this "
            f"screen:\n{listing}\n"
            f"Fix (either one):\n"
            f"  - set {env_override_var(table, field)}=S<n> in the "
            f"environment and re-run -- no rebuild needed; or\n"
            f"  - add PUSH_BUTTONS[({table!r}, {field!r})] = "
            f"push_button_id('S<n>') in \n"
            f"If every label above is '(no label found)', run the app's "
            f"Diagnose button and send the log -- the raw screen dump names "
            f"which S<n> is which.")
    resolved = hits[0]["push_id"]
    PUSH_BUTTONS[key] = resolved
    if log:
        log(f"SAP: resolved {field} on {table} -> {hits[0]['param']} "
            f"by on-screen {matched_on} "
            f"({hits[0][matched_on]!r})")
    return resolved


class Cancelled(Exception):
    """Raised inside a worker step when the user has pressed Stop."""


# ---------------------------------------------------------------------------
# Key normalization (ported from the notebook -- same rules, single copy)
# ---------------------------------------------------------------------------

# Fix E: only expand scientific notation when the whole string matches it,
# not any string that happens to contain the letter 'E'. Otherwise object
# numbers like "1E2000" get mangled to "100" + trailing garbage.
_SCI_NOTATION_RE = re.compile(r"^-?\d+(\.\d+)?[eE][+-]?\d+$")

# Rows whose primary key cell equals one of these (case-insensitive) are
# not sent to SAP as filter values, mirroring the notebook's paste-side
# guard: `if v != "" and v.upper() not in {"NOT FOUND","NOTFOUND"}`.
# Note: the notebook's WRITE-BACK loop still clears their output cells (via
# ClearContents + the non-match else branch), so this refactor does too --
# skip rows are treated as non-matches for the write path, not preserved.
_SKIP_KEY_MARKERS = frozenset({"NOT FOUND", "NOTFOUND"})

# Item 2: maximum filter values pasted into one SAP query. Larger key lists
# are split into multiple navigate->paste->execute->export rounds and the
# exports concatenated before matching. 2000 keeps the multi-select dialog
# and the ALV export comfortably inside SAP's practical limits.
SAP_FILTER_CHUNK_SIZE = 2000


def _clean_cell(value) -> str:
    """Turn a raw Excel/pandas value into a stripped string, treating None,
    NaN, and pywintypes cell-error ints as empty.
    - Fix B: str(float('nan')) is 'nan' -- filter NaN before it becomes a key.
    - Fix G: #N/A / #REF! / #VALUE! come back from Excel COM as large-negative
      int codes (e.g. -2146826281). Treat those as blanks too.
    """
    if value is None:
        return ""
    if isinstance(value, float):
        # NaN check without importing math.isnan (works for float NaN)
        if value != value:
            return ""
    if isinstance(value, int) and not isinstance(value, bool) and value < -2_000_000_000:
        return ""
    return str(value).strip()


def _canonicalize_number_shape(txt: str) -> str:
    """Shared body of normalize_key / clean_numeric_for_sap."""
    for ch in (" ", "\t", chr(160), "'", ","):
        txt = txt.replace(ch, "")
    if _SCI_NOTATION_RE.match(txt):
        try:
            txt = format(Decimal(txt), "f")
        except InvalidOperation:
            pass
    if "." in txt:
        left, right = txt.split(".", 1)
        if right == "" or set(right) <= {"0"}:
            txt = left
    return txt


def normalize_key(value) -> str:
    """Aggressively normalize an Excel value so it matches SAP's key form.

    - drop None, NaN, and Excel cell-error codes
    - strip whitespace / NBSP / apostrophes / commas
    - collapse scientific notation ONLY when the whole string is sci-notation
    - drop trailing '.0' / '.00' / '.'
    - drop leading zeros (keep at least one char)

    The notebook applied different (weaker) rules per step, but every rule
    used here is applied identically to both the Excel side and the SAP side
    inside `_build_lookup`, so this can only add matches, never subtract.
    """
    txt = _clean_cell(value)
    if not txt:
        return ""
    txt = _canonicalize_number_shape(txt)
    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]
    return txt


def clean_numeric_for_sap(value) -> str:
    """Like normalize_key but preserves leading zeros -- used when pasting
    numeric identifiers into SAP where SAP itself will canonicalize them.
    """
    txt = _clean_cell(value)
    if not txt:
        return ""
    return _canonicalize_number_shape(txt)


def _is_skip_key_cell(value) -> bool:
    """True when this Excel key cell should NOT be pasted into SAP's filter
    dialog (blank, or one of the 'already resolved' markers). Matches the
    notebook's paste-side guard. Skipped rows are still processed by the
    write-back loop as non-matches.
    """
    txt = _clean_cell(value)
    if not txt:
        return True
    return txt.upper() in _SKIP_KEY_MARKERS


# ---------------------------------------------------------------------------
# Workflow definitions
# ---------------------------------------------------------------------------

@dataclass
class ExtraOutput:
    """A per-step output column that lives outside the main output block.

    `sap_col=None` means "no SAP source" -- the column is only cleared on a
    non-match (used to mirror the notebook's TO Step 3 behavior on column J,
    which is cleared when a row fails to match but preserved when it does).

    `preserve_on_nonmatch=True` means "on non-match, do not touch this cell"
    (mirrors TO Step 2's column A: notebook comment "Do not touch Column A
    if there is no match").

    Skip rows (blank / "NOT FOUND" key) are treated identically to non-match
    rows for the extras, matching the notebook where the write-back loop
    iterates every row and only the key check inside the loop distinguishes
    match vs. else (skip rows fall into the else branch).
    """
    excel_col: int
    sap_col: str | None = None
    preserve_on_nonmatch: bool = False
    # Optional per-value transform applied to the fetched SAP value before
    # writing (and before publishing for downstream steps). Used to derive
    # the reservation pair out of the unloading point -- the job the Excel
    # template used to do with a formula.
    transform: Callable[[str], str] | None = None


@dataclass
class LookupStep:
    name: str                     # human label for logs
    sap_table: str                # e.g. "LTAP"
    push_button_field: str        # e.g. "TO_NUMBER" -- indexes PUSH_BUTTONS
    key_columns: list[int]        # 1-based Excel column indices used as key
    key_joiner: str = "|"         # how multi-column keys are joined
    sap_key_columns: list[str] = field(default_factory=list)     # SAP df cols composing the match key
    sap_output_columns: list[str] = field(default_factory=list)  # SAP df cols we copy to Excel
    excel_output_columns: list[int] = field(default_factory=list)  # 1-based Excel columns to write
    extras: list[ExtraOutput] = field(default_factory=list)      # per-cell exceptions to the main block
    # If set, write header + composite match key (normalize_key of every part
    # joined by key_joiner) to this 1-based column, rows 1..last_row.
    # Notebook TO-2 does this for column P as an audit trail.
    match_key_column: int | None = None
    match_key_header: str = "Excel Match Key Used"


# The ESA unloading point (LTAP ABLAD) encodes the reservation pair:
# 10-digit zero-padded reservation number + 4-digit item. Example from the
# live workbook: ABLAD '05175455500001' -> N 517545550, O 1;
# '05175456020012' -> N 517545602, O 12. The Excel template used to split
# this with a FORMULA in columns N/O -- which the app cannot rely on: all
# writes are deferred to the end of the run, so step 2 would read N/O
# before M exists in the sheet, and the formula itself is easily lost when
# a sheet is copied. The split lives here instead; anything that does not
# look like the encoded form (blank, 'NOT FOUND', a plain dock name)
# yields "", which step 2 then skips as a row without a reservation.

def _reservation_from_unloading_point(v) -> str:
    s = str(v or "").strip()
    if len(s) < 5 or not s.isdigit():
        return ""
    return s[:-4].lstrip("0") or "0"


def _item_from_unloading_point(v) -> str:
    s = str(v or "").strip()
    if len(s) < 5 or not s.isdigit():
        return ""
    return s[-4:].lstrip("0") or "0"


WORKFLOWS = {
    "TO": [
        LookupStep(
            name="LTAP -> Unloading Point",
            sap_table="LTAP",
            push_button_field="TO_NUMBER",
            key_columns=[11],                     # K
            sap_key_columns=["TANUM"],            # LTAP TO number field
            sap_output_columns=["ABLAD"],
            excel_output_columns=[13],            # M
            extras=[
                # The template formula, moved into the pipeline: N and O
                # are split out of the unloading point so step 2 can key on
                # them (in memory -- no Excel round trip) and the sheet
                # still shows them for auditing. Non-match rows get ""
                # like every other output.
                ExtraOutput(excel_col=14, sap_col="ABLAD",
                            transform=_reservation_from_unloading_point),
                ExtraOutput(excel_col=15, sap_col="ABLAD",
                            transform=_item_from_unloading_point),
            ],
        ),
        LookupStep(
            # Keyed on the reservation pair, exactly like the ESA
            # developer's own notebook (his code is the spec, 2026-08). A
            # 2026-08 rework briefly keyed this on the TO number instead
            # (905147d); reverted -- the reservation path is the one proven
            # in the field, N/O are operator-provided inputs in the live
            # workbook, and preserving A on non-match is what lets the
            # NOTIF process later fill the notification-only rows (the
            # either/or model: most rows have a notification but no TO).
            name="Z50CFG_ENG_CRNT (Reservation) -> QMNUM/OBJNR/DISP",
            sap_table="Z50CFG_ENG_CRNT",
            push_button_field="RSNUM",
            key_columns=[14, 15],                 # N | O
            sap_key_columns=["RSNUM", "RSPOS"],
            sap_output_columns=["OBJNR", "DISP_MATNR", "DISP_QTY"],
            excel_output_columns=[3, 4, 5],       # C, D, E
            extras=[
                # Notebook rule: A gets QMNUM on match but MUST NOT be
                # touched on non-match ("Do not touch Column A if there is
                # no match") -- the untouched rows carry the pre-existing
                # notification numbers the NOTIF process keys on.
                ExtraOutput(excel_col=1, sap_col="QMNUM", preserve_on_nonmatch=True),
            ],
            match_key_column=16,                  # P: audit-trail composite key
        ),
        LookupStep(
            name="Z50CFG_ENG_VALD -> Section/Module/Description/SalesDoc",
            sap_table="Z50CFG_ENG_VALD",
            push_button_field="OBJNR",
            key_columns=[3],                      # C
            sap_key_columns=["OBJNR"],
            # Notebook TO-3 sap_data_map only carries these 4 (LID is NOT
            # populated on match). Column J is cleared in the non-match
            # branch and left alone on match -- modeled by the extras entry.
            sap_output_columns=["Z_SECTION", "Z_MODULE", "DESCRIPT", "SALES_ORDER"],
            excel_output_columns=[6, 7, 8, 9],    # F..I
            extras=[ExtraOutput(excel_col=10)],   # J: preserve on match, clear on non-match/skip
        ),
    ],
    "NOTIF": [
        LookupStep(
            name="Z50CFG_ENG_CRNT (Notification) -> OBJNR/DISP_MATNR/DISP_QTY",
            sap_table="Z50CFG_ENG_CRNT",
            push_button_field="QMNUM",
            key_columns=[1],                      # A
            sap_key_columns=["QMNUM"],
            sap_output_columns=["OBJNR", "DISP_MATNR", "DISP_QTY"],
            excel_output_columns=[3, 4, 5],       # C, D, E
        ),
        LookupStep(
            name="Z50CFG_ENG_VALD -> Section/Module/Description/SalesDoc/LID",
            sap_table="Z50CFG_ENG_VALD",
            push_button_field="OBJNR",
            key_columns=[3],                      # C
            sap_key_columns=["OBJNR"],
            # Notebook NOTIF-2 fetches LID from SAP but never writes it (bug
            # -- match branch writes only F..I). The completion message and
            # this project's README both say LID -> J, so we treat that as
            # the user's true intent and route LID through an ExtraOutput.
            # ClearContents range therefore matches notebook (F..I only).
            sap_output_columns=["Z_SECTION", "Z_MODULE", "DESCRIPT", "SALES_ORDER"],
            excel_output_columns=[6, 7, 8, 9],    # F..I
            extras=[ExtraOutput(excel_col=10, sap_col="LID", preserve_on_nonmatch=False)],
        ),
    ],
}


# ---------------------------------------------------------------------------
# Event callback protocol
# ---------------------------------------------------------------------------

# on_event(kind, payload):
#   ("log", (message: str, level: "info"|"ok"|"warn"|"error"))
#   ("status", message: str)
#   ("progress", fraction: float 0..1)
#   ("popup", {"kind": "step"|"info"|"error", "title": str, "message": str,
#              "ack": threading.Event|None, "result": {"proceed": bool}|None})
#     The original notebook spoke to its operator through blocking message
#     boxes -- a counts popup after every step, an error popup on failure --
#     and the ESA operator troubleshoots by screenshotting those together
#     with the SAP screen. These events reproduce that behavior. "step"
#     popups carry an ack Event: the handler MUST show OK/Cancel, write the
#     choice into result["proceed"], and set ack. "info"/"error" popups are
#     fire-and-forget.
#   ("done", ok: bool)

EventFn = Callable[[str, object], None]


# ---------------------------------------------------------------------------
# Per-run file log -- persistent record so a mid-run crash can be diagnosed
# after the fact even if the GUI closed. Path is echoed into the GUI log at
# run start.
# ---------------------------------------------------------------------------
_LOG_FH = None       # file handle
_LOG_FILE = ""       # current log path


def _open_run_log() -> str:
    """Open a timestamped log file for this run. Never raises; returns the
    path (empty on failure). Also prunes older runs to keep at most 20 files.
    """
    global _LOG_FH, _LOG_FILE
    _LOG_FH = None
    _LOG_FILE = ""
    try:
        base = os.environ.get("LOCALAPPDATA") or tempfile.gettempdir()
        log_dir = os.path.join(base, "esa-lookup", "logs")
        os.makedirs(log_dir, exist_ok=True)
        # Prune to the 20 most recent .log files.
        try:
            existing = sorted(
                (os.path.join(log_dir, n) for n in os.listdir(log_dir) if n.endswith(".log")),
                key=os.path.getmtime,
            )
            for old in existing[:-19]:
                try:
                    os.remove(old)
                except OSError:
                    pass
        except OSError:
            pass
        stamp = time.strftime("%Y%m%d-%H%M%S")
        path = os.path.join(log_dir, f"esa-lookup-{stamp}.log")
        _LOG_FH = open(path, "w", encoding="utf-8", buffering=1)
        _LOG_FILE = path
        return path
    except Exception:
        _LOG_FH = None
        _LOG_FILE = ""
        return ""


def _close_run_log() -> None:
    global _LOG_FH
    if _LOG_FH is not None:
        try:
            _LOG_FH.close()
        except Exception:
            pass
    _LOG_FH = None


def _file_log(level: str, msg: str) -> None:
    """Best-effort write to the current run's log file. Never raises."""
    if _LOG_FH is None:
        return
    try:
        _LOG_FH.write(
            f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {level.upper():5s} {msg}\n"
        )
    except Exception:
        pass


def _file_log_traceback() -> None:
    """Persist a full traceback of the current exception to the log file
    only -- the GUI shows the one-line summary, the file gets the full chain
    so a post-mortem can see what really happened."""
    if _LOG_FH is None:
        return
    try:
        _LOG_FH.write(traceback.format_exc())
        _LOG_FH.write("\n")
    except Exception:
        pass


def _log(on_event: EventFn, msg: str, level: str = "info") -> None:
    _file_log(level, msg)
    on_event("log", (msg, level))


def _status(on_event: EventFn, msg: str) -> None:
    _file_log("STAT", f"[status] {msg}")
    on_event("status", msg)


def _progress(on_event: EventFn, frac: float) -> None:
    # Progress ticks are too noisy for the file log.
    on_event("progress", max(0.0, min(1.0, frac)))


def _popup(on_event: EventFn, kind: str, title: str, message: str,
           ack=None, result=None) -> None:
    _file_log("POPUP", f"[{kind}] {title}: " + message.replace("\n", " | "))
    on_event("popup", {"kind": kind, "title": title, "message": message,
                       "ack": ack, "result": result})


def _step_popup(on_event: EventFn, stop: threading.Event,
                title: str, message: str) -> None:
    """Blocking per-step popup, the original notebook's rhythm: show the
    counts, wait for the operator's OK before touching SAP again -- so they
    can inspect the result grid still on screen -- or Cancel to stop the
    run (the workbook is untouched during the fetch phase).

    The wait polls so a Stop pressed through other means still interrupts.
    If the event handler never acks (headless caller that ignores popups),
    the run would wait forever -- which is why step popups are opt-in via
    RunConfig.step_popups and every shipped handler acks.
    """
    ack = threading.Event()
    result = {"proceed": True}
    _popup(on_event, "step", title, message, ack=ack, result=result)
    while not ack.wait(0.1):
        if stop.is_set():
            raise Cancelled()
    if not result["proceed"]:
        stop.set()
        raise Cancelled()


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def _col_letter(idx: int) -> str:
    """1-based column index -> Excel letter (only up to ZZ, plenty here)."""
    result = ""
    n = idx
    while n > 0:
        n, r = divmod(n - 1, 26)
        result = chr(65 + r) + result
    return result


def _range(col_start: int, col_end: int, row_start: int, row_end: int) -> str:
    return (
        f"{_col_letter(col_start)}{row_start}:{_col_letter(col_end)}{row_end}"
    )


# Fix 3: SAP ALV exports typically use the ALV column TITLE, which is often a
# short description rather than the technical field name. Try the technical
# name first, then any known aliases. Extend per-site if needed.
SAP_COLUMN_ALIASES: dict[str, list[str]] = {
    "TANUM":       ["TANUM", "TO Number", "TO No.", "Transfer Order", "TrfOrd", "TrfOrdNo"],
    # "Unl. Point" is what the ESA ZTBV layout actually prints for ABLAD --
    # the ALV shows English short labels, not technical names.
    "ABLAD":       ["ABLAD", "Unl. Point", "Unloading Point", "UnloadPt"],
    "QMNUM":       ["QMNUM", "Notifctn", "Notification", "Notification No", "Notification Number"],
    "RSNUM":       ["RSNUM", "Reserv.No.", "Reservation", "Reservation No", "Reservation Number", "Res.Number"],
    # ORDER MATTERS. The Z50CFG_ENG_CRNT layout carries BOTH pairs side by
    # side: 'Reserv.No.' + 'Itm' (the reservation) and 'TO Number' + 'Item'
    # (the transfer order). RSPOS is the RESERVATION item, so 'Itm' has to
    # be tried before the generic 'Item' -- otherwise RSPOS silently binds
    # to the TO item, resolution "succeeds", and step 2 keys on a mismatched
    # pair that quietly matches nothing.
    "RSPOS":       ["RSPOS", "Itm", "Res.Item", "Item No", "Item Number", "Item"],
    "OBJNR":       ["OBJNR", "Object number", "Object Number", "Obj.Number", "Object No"],
    "DISP_MATNR":  ["DISP_MATNR", "Disp Matl", "Disp Material", "Disposition Material", "Disp.Material"],
    "DISP_QTY":    ["DISP_QTY", "Disp Qty", "Disposition Qty", "Disposition Quantity", "Disp.Qty"],
    "Z_SECTION":   ["Z_SECTION", "Section"],
    "Z_MODULE":    ["Z_MODULE", "Module"],
    "DESCRIPT":    ["DESCRIPT", "Description", "Descr."],
    "SALES_ORDER": ["SALES_ORDER", "Sales Order", "Sales Doc.", "Sales Doc", "Sales Doc. No."],
    "LID":         ["LID"],
}


def _norm_col(name) -> str:
    """Fold an ALV column title for tolerant comparison: case, spaces, and
    the punctuation SAP sprinkles through its abbreviations all ignored, so
    "Unl. Point" / "Unl.Point" / "UNL POINT" are one name.
    """
    return re.sub(r"[\s._\-/]+", "", str(name)).lower()


def _resolve_column(df: pd.DataFrame, canonical: str) -> str | None:
    """Return whichever of `canonical`'s alias names is present in df, or None.

    Exact match first, then the folded comparison -- an ALV variant that
    prints "Unl.Point" should still resolve an alias spelled "Unl. Point"
    without needing a separate entry for every punctuation variant.
    """
    aliases = SAP_COLUMN_ALIASES.get(canonical, [canonical])
    for name in aliases:
        if name in df.columns:
            return name
    folded: dict[str, list[str]] = {}
    for actual in df.columns:
        folded.setdefault(_norm_col(actual), []).append(actual)
    for name in aliases:
        hits = folded.get(_norm_col(name))
        if hits:
            return hits[0]
    return None


# ---------------------------------------------------------------------------
# Reading whatever SAP actually exported
# ---------------------------------------------------------------------------

# SAP names every ALV export ".xlsx", but only the `&XXL` path really writes
# one. When `&XXL` is unavailable (older SAP GUI build, or no Excel/XXL
# frontend integration -- it fails with "The control could not be found"),
# `export_alv_to_file` falls back to `&PC` "Save as Local File",
# which writes a DELIMITED TEXT file under that same .xlsx name. Feeding
# that to pd.read_excel dies with "Excel file format cannot be determined,
# you must specify an engine manually".
#
# Independently, on sites whose Office integration is enabled, the `&XXL`
# path itself can write an MHTML-wrapped <table> under the .xlsx name --
# openpyxl rejects that one with "not a zip file".
#
# So the extension tells us nothing: sniff the real format from the leading
# bytes and dispatch on that.

_ZIP_MAGIC = b"PK\x03\x04"          # .xlsx / OOXML (a zip container)
_OLE2_MAGIC = b"\xd0\xcf\x11\xe0"   # legacy .xls (BIFF8 inside OLE2)


def _decode_sap_text(raw: bytes) -> str:
    """Decode an SAP text export: honour the BOM SAP writes on Unicode
    systems, else fall back to the Windows frontend codepage.
    """
    for bom, enc in ((b"\xff\xfe\x00\x00", "utf-32"),
                     (b"\x00\x00\xfe\xff", "utf-32"),
                     (b"\xff\xfe", "utf-16"),
                     (b"\xfe\xff", "utf-16"),
                     (b"\xef\xbb\xbf", "utf-8-sig")):
        if raw.startswith(bom):
            return raw.decode(enc)
    for enc in ("utf-8", "cp1252"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("latin-1", "replace")


def _sap_text_to_frame(text: str) -> pd.DataFrame:
    """Parse an SAP `&PC` export -- either the tab-delimited "Spreadsheet"
    format or the pipe-ruled "unconverted" list format -- into a DataFrame.

    Every cell stays a string. SAP already formatted the values the way the
    ALV displayed them, and keeping them verbatim preserves leading zeros on
    material / TO numbers that pandas' numeric inference would eat.
    `normalize_key` canonicalizes both sides of the join anyway, so string
    cells match exactly the rows numeric cells would.
    """
    lines = []
    for ln in text.splitlines():
        s = ln.strip()
        # skip blanks and ALV list rulers: "--------" / "|----+----|"
        if not s or set(s) <= set("-+| "):
            continue
        lines.append(ln)
    if not lines:
        return pd.DataFrame()

    if sum(ln.count("|") for ln in lines) > sum(ln.count("\t") for ln in lines):
        rows = [[c.strip() for c in ln.strip().strip("|").split("|")]
                for ln in lines]
    else:
        rows = [[c.strip() for c in ln.split("\t")] for ln in lines]

    # SAP prefixes some list exports with report title / run-date lines that
    # carry fewer fields than the table. The real header row is the first one
    # with the table's own field count (the most common width).
    widths = [len(r) for r in rows]
    modal = max(set(widths), key=widths.count)
    first = widths.index(modal)
    # Rows of a different width are footers ("3 record(s) selected") or wrapped
    # lines; dropping them can only UNDER-count, which the caller's
    # grid_rows comparison turns into a loud error rather than silent loss.
    rows = [r for r in rows[first:] if len(r) == modal]

    header = [h or f"Unnamed: {i}" for i, h in enumerate(rows[0])]
    return pd.DataFrame(rows[1:], columns=header)


def _read_sap_export(path: str, log=None) -> pd.DataFrame:
    """Read the file `export_alv_to_file` produced, whatever format
    SAP actually chose for it.
    """
    with open(path, "rb") as fh:
        raw = fh.read()
    head = raw[:2048]

    # dtype=str everywhere: pandas' numeric inference would eat the
    # leading zeros of encoded values -- the ESA unloading point
    # '00000001000001' must not come back as the number 1000001, or the
    # reservation split (and any key match on it) silently corrupts.
    # normalize_key canonicalizes both sides of every join, so all-string
    # frames match exactly the rows inferred-numeric frames would.
    if head.startswith(_ZIP_MAGIC):
        return pd.read_excel(path, engine="openpyxl", dtype=str)

    if head.startswith(_OLE2_MAGIC):
        try:
            return pd.read_excel(path, engine="xlrd", dtype=str)
        except ImportError as e:
            raise RuntimeError(
                f"SAP exported a legacy .xls workbook ({path}); reading it "
                f"needs the 'xlrd' package -- run: pip install xlrd"
            ) from e

    # MHTML / HTML: some sites' Office integration makes "Export as
    # Spreadsheet" write an MHTML-wrapped <table> under the .xlsx name.
    # The MIME preamble has to be sliced off before read_html sees it.
    lowered = head.lower()
    payload = None
    if b"mime-version:" in lowered or b"content-location:" in lowered:
        i = raw.lower().find(b"<html")
        if i < 0:
            raise RuntimeError(
                f"SAP export {path} looks like MHTML but no <html> body was "
                f"found.")
        payload = raw[i:]
    elif b"<html" in lowered or b"<table" in lowered:
        payload = raw

    if payload is not None:
        # Must be a file-like object: pandas >=3 treats a bytes/str argument
        # as a PATH, so passing the markup itself raises FileNotFoundError.
        try:
            tables = pd.read_html(io.BytesIO(payload))
        except ImportError as e:
            raise RuntimeError(
                f"SAP exported an HTML/MHTML table ({path}); reading it needs "
                f"the 'lxml' package -- run: pip install lxml"
            ) from e
        if not tables:
            raise RuntimeError(
                f"SAP export {path}: no <table> found in the HTML body.")
        return tables[0]

    text = _decode_sap_text(raw)
    if log:
        log("SAP: export is a delimited text file (&PC fallback), not xlsx -- "
            "parsing as text")
    df = _sap_text_to_frame(text)
    # We only get here for a grid that reported rows (the caller skips empty
    # grids), so a frame with no data rows means the file is not a delimited
    # export at all -- say so instead of failing later on missing columns.
    if df.empty:
        raise RuntimeError(
            f"Could not parse the SAP export at {path}: it is neither an "
            f"xlsx/xls workbook nor a recognizable delimited text export. "
            f"Open it manually to see what ZTBV actually wrote."
        )
    return df


def _build_lookup(
    df: pd.DataFrame,
    key_cols: list[str],
    value_cols: list[str],
) -> tuple[dict, int, int]:
    """Return (lookup, dup_count, blank_key_count) built from df.

    Resolves each canonical SAP field name (e.g. "TANUM") to whichever alias
    (e.g. "TO Number") the ALV export actually used.

    Fix C: duplicate composite keys keep the FIRST occurrence and increment
    dup_count so the caller can warn -- overwriting silently loses data when
    a lookup table legitimately has multiple rows per key.

    Fix D: rows where ANY key part is blank are skipped, not just rows where
    ALL parts are blank. A composite of "12345|" would otherwise falsely
    match every SAP row with the same first part and a blank second.
    """
    needed = list(dict.fromkeys(key_cols + value_cols))
    resolved: dict[str, str] = {}
    missing: list[str] = []
    for c in needed:
        r = _resolve_column(df, c)
        if r is None:
            missing.append(c)
        else:
            resolved[c] = r
    if missing:
        # Name the closest present titles: the ALV prints English short
        # labels ("Unl. Point" for ABLAD), so the field is usually right
        # there under a name no alias lists yet. Guessing it here saves a
        # round trip to whoever is standing at the customer's machine.
        hints = []
        for c in missing:
            near = difflib.get_close_matches(
                c, [str(x) for x in df.columns], n=3, cutoff=0.4)
            for alias in SAP_COLUMN_ALIASES.get(c, []):
                near += difflib.get_close_matches(
                    alias, [str(x) for x in df.columns], n=3, cutoff=0.6)
            near = list(dict.fromkeys(near))[:3]
            if near:
                hints.append(f"  {c}: closest titles present are {near}")
        raise RuntimeError(
            "SAP export is missing these expected columns: "
            f"{missing}\n"
            f"Columns present in the export: {list(df.columns)}\n"
            + ("Did you mean:\n" + "\n".join(hints) + "\n" if hints else "")
            + "Fix: edit the ALV layout in ZTBV so each missing field is shown "
            "(prefer 'Technical Name' as the column title) and re-save the "
            "default variant, OR add another alias to SAP_COLUMN_ALIASES in "
            "pipeline.py."
        )

    # A title repeated in the layout makes df[title] a DataFrame slice, so
    # row[title] is a Series -- normalize_key would stringify the whole
    # Series and pd.isna would raise "truth value is ambiguous". The ESA
    # LTAP layout repeats 'Item', 'Typ', 'Sec' and 'B.pos' several times,
    # so fail loudly here rather than write nonsense into the workbook.
    titles = [str(x) for x in df.columns]
    ambiguous = {c: r for c, r in resolved.items() if titles.count(str(r)) > 1}
    if ambiguous:
        raise RuntimeError(
            "These SAP fields resolved to a column title that appears more "
            "than once in the export, so the right one cannot be told apart:\n"
            + "\n".join(f"  {c} -> {r!r} (appears {titles.count(str(r))} times)"
                        for c, r in ambiguous.items())
            + "\nFix: edit the ALV layout in ZTBV to show 'Technical Name' as "
            "the column title (which is unique per field), or remove the "
            "duplicate columns from the layout and re-save the variant."
        )

    # Two SAP fields landing on one column means an alias list is too greedy
    # (the way RSPOS's generic "Item" once outranked the reservation's own
    # "Itm"). Resolution would "succeed" and the step would then key on a
    # field it was never meant to read, so refuse rather than guess.
    collisions: dict[str, list[str]] = {}
    for c, r in resolved.items():
        collisions.setdefault(str(r), []).append(c)
    clashing = {r: cs for r, cs in collisions.items() if len(cs) > 1}
    if clashing:
        raise RuntimeError(
            "These SAP fields all resolved to the SAME export column, so at "
            "least one of them is bound to the wrong field:\n"
            + "\n".join(f"  {cs} -> {r!r}" for r, cs in clashing.items())
            + "\nFix: tighten the alias lists in SAP_COLUMN_ALIASES so each "
            "field matches its own column title first."
        )
    out: dict[str, dict] = {}
    dup_count = 0
    blank_key_count = 0
    for _, row in df.iterrows():
        parts = [normalize_key(row[resolved[c]]) for c in key_cols]
        if any(p == "" for p in parts):
            blank_key_count += 1
            continue
        composite = "|".join(parts)
        if composite in out:
            dup_count += 1
            continue  # keep first occurrence -- do not silently overwrite
        out[composite] = {
            c: ("" if pd.isna(row[resolved[c]]) else row[resolved[c]])
            for c in value_cols
        }
    return out, dup_count, blank_key_count


class VirtualSheet:
    """Gen 4 key-column resolver.

    Columns PRODUCED by an earlier step in this run are served from memory
    (`publish`); everything else is read from the workbook. This is what
    lets TO step 3 key off the OBJNR values step 2 just fetched, without an
    intermediate Excel write/read round-trip. A column no step has produced
    falls back to the workbook -- which also preserves the Gen 2/3 behavior
    for a skipped/no-op producer step (the consumer then sees whatever was
    already in the sheet, exactly as before).

    `last_row` mirrors Excel's End(xlUp) for in-memory columns by taking the
    last row whose value is non-empty (the Excel path wrote "" into
    non-matched cells, which End(xlUp) skips over as blanks).
    """

    def __init__(self, sheet):
        self.sheet = sheet
        self._mem: dict[int, list] = {}

    def publish(self, col: int, values: list) -> None:
        """Register `values` as the would-be content of rows 2..2+len-1."""
        self._mem[col] = values

    def is_virtual(self, col: int) -> bool:
        return col in self._mem

    def last_row(self, col: int) -> int:
        if col in self._mem:
            last = 1
            for i, v in enumerate(self._mem[col]):
                if _clean_cell(v) != "":
                    last = i + 2
            return last
        return last_row_in_column(self.sheet, col)

    def get_col(self, col: int, last_row: int) -> list:
        """Values for rows 2..last_row, padded with '' beyond available data
        (Excel semantics: cells below a produced/short column read as blank
        because the producing step's clear ran down to Rows.Count)."""
        n = max(0, last_row - 1)
        if col in self._mem:
            vals = list(self._mem[col][:n])
        else:
            rng = f"{_col_letter(col)}2:{_col_letter(col)}{last_row}"
            rows = read_range_2d(self.sheet, rng)
            vals = [(r[0] if r else "") for r in rows]
        return vals + [""] * (n - len(vals))


@dataclass
class StepResult:
    """Everything a fetched step carries into the deferred write-back."""
    step: LookupStep
    step_index: int
    last_row: int
    excel_keys: list[str]
    entries_by_row: list[dict | None]
    n_skipped: int
    matched: int
    unmatched: int
    sap_rows: int = 0     # total rows SAP's grid reported across all chunks


def _fetch_step(
    step: LookupStep,
    vsheet: VirtualSheet,
    xl: ExcelCtx,
    sap: SapSession,
    tmp_dir: str,
    on_event: EventFn,
    step_index: int,
    total_steps: int,
    stop: threading.Event,
    step_popups: bool = False,
) -> StepResult | None:
    """Run the SAP side of one lookup step and match it against the step's
    key column -- WITHOUT touching the workbook. Returns None when the step
    is a no-op (empty key column / no usable values).

    Gen 4: key columns written by an earlier step in this run resolve from
    memory via `vsheet`, so steps chain without an Excel round-trip.
    """

    # Progress: total_steps fetch segments + one final write segment.
    n_segments = total_steps + 1

    def sub_progress(sub_i: int, sub_n: int = 6):
        _progress(on_event, (step_index + sub_i / sub_n) / n_segments)

    step_started = time.time()
    if stop.is_set():
        raise Cancelled()

    # --- 1. Resolve key columns (memory first, Excel otherwise) ----------
    _status(on_event, f"[{step_index + 1}/{total_steps}] {step.name}: reading keys")
    _log(on_event, f"--- Step {step_index + 1}/{total_steps}: {step.name}  "
                    f"(table={step.sap_table}, key_cols={step.key_columns})")

    # Fix F: each step derives its own last_row from its own primary key
    # column (matches notebook, which re-computes last_row per step). A short
    # column later in the workflow doesn't process phantom rows from a
    # longer column earlier in the workflow.
    primary_col = step.key_columns[0]
    if vsheet.is_virtual(primary_col):
        _log(on_event,
             f"key column {_col_letter(primary_col)} resolved from an earlier "
             f"step's in-memory result (no Excel round-trip)")
    last_row = vsheet.last_row(primary_col)
    if last_row < 2:
        _log(on_event,
             f"No data below the header in column {_col_letter(primary_col)} "
             f"(#{primary_col}); step skipped", "warn")
        _progress(on_event, (step_index + 1) / n_segments)
        if step_popups:
            _step_popup(
                on_event, stop,
                f"Step {step_index + 1} of {total_steps} skipped",
                f"{step.name}\n\n"
                f"No data below the header in column "
                f"{_col_letter(primary_col)} -- nothing to look up.\n\n"
                f"OK = continue with the next step, Cancel = stop the run.")
        return None

    key_vals = {c: vsheet.get_col(c, last_row) for c in step.key_columns}
    n_rows = last_row - 1
    sub_progress(1)

    # excel_keys is the normalized composite for every row -- used for both
    # lookup matching and (when match_key_column is set) the P-column audit
    # write. Notebook builds the identical string via `normalize_key(...)|
    # normalize_key(...)` inside its per-row loop. Parts are joined in
    # ascending column order (matches the Gen 2/3 offset-sorted read).
    ordered_cols = sorted(step.key_columns)
    excel_keys = [
        step.key_joiner.join(normalize_key(key_vals[c][i]) for c in ordered_cols)
        for i in range(n_rows)
    ]
    # Skip flags: only used to (a) exclude from SAP paste, (b) count for the
    # log summary. The write-back treats skip rows as non-matches, which
    # is what the notebook does implicitly (its else branch fires for any
    # composite key that's not in the SAP result set, including "NOTFOUND|").
    # ESA rule: a row is processed only when EVERY part of its key has a
    # value. A sheet legitimately mixes rows that carry a reservation with
    # rows that do not -- the complete ones must still be processed, and the
    # incomplete ones are skipped rather than failing anything. Previously
    # only the PRIMARY cell was tested, so "N filled, O blank" was sent to
    # SAP and then reported as unmatched.
    skip_flags = [
        any(_is_skip_key_cell(key_vals[c][i]) for c in ordered_cols)
        for i in range(n_rows)
    ]

    # Say how many rows were dropped for a PARTIAL key specifically -- those
    # look like usable rows to the operator, so silently folding them into
    # the skip count would hide a half-filled sheet.
    if len(ordered_cols) > 1:
        partial = [
            i for i in range(n_rows)
            if skip_flags[i] and not _is_skip_key_cell(key_vals[primary_col][i])
        ]
        if partial:
            blank_cols = sorted({
                c for c in ordered_cols for i in partial
                if _is_skip_key_cell(key_vals[c][i])
            })
            _log(on_event,
                 f"note: {len(partial)} row(s) have column "
                 f"{_col_letter(primary_col)} filled but column(s) "
                 f"{', '.join(_col_letter(c) for c in blank_cols)} blank -- "
                 f"skipped, since a complete key needs every one of "
                 f"{[_col_letter(c) for c in ordered_cols]}. Rows with a "
                 f"complete key are unaffected.")

    unique_paste_values: list[str] = []
    seen: set[str] = set()
    for i in range(n_rows):
        if skip_flags[i]:
            continue
        # For paste, use the FIRST key column value (SAP filter dialog is
        # single-column). Multi-column keys still work because SAP returns
        # the full result set and we filter by composite key on our side.
        v = clean_numeric_for_sap(key_vals[primary_col][i])
        if v and v not in seen:
            seen.add(v)
            unique_paste_values.append(v)

    n_skipped = sum(skip_flags)
    if n_skipped:
        _log(on_event,
             f"note: {n_skipped} row(s) in column {_col_letter(primary_col)} "
             f"are blank or 'NOT FOUND' -- not sent to SAP; their output "
             f"cells will be cleared just like any other non-match")

    if not unique_paste_values:
        _log(on_event,
             f"No usable values in column(s) {step.key_columns} "
             f"(after skipping blanks / 'NOT FOUND'); step is a no-op",
             "warn")
        _progress(on_event, (step_index + 1) / n_segments)
        if step_popups:
            _step_popup(
                on_event, stop,
                f"Step {step_index + 1} of {total_steps} skipped",
                f"{step.name}\n\n"
                f"Column(s) "
                f"{'/'.join(_col_letter(c) for c in step.key_columns)} hold "
                f"no usable keys (all blank or 'NOT FOUND') -- nothing was "
                f"sent to SAP.\n\n"
                f"OK = continue with the next step, Cancel = stop the run.")
        return None

    _log(on_event, f"{len(unique_paste_values)} unique key(s) will be sent to SAP")
    sub_progress(2)

    # --- 2. Query SAP in chunks: navigate, paste, execute, verify, export
    # Item 2: large key lists are split so a single oversized multi-select
    # paste can't overload the selection screen or produce an unmanageable
    # export. Each chunk is a full navigate->paste->execute->export round;
    # chunk results are concatenated before matching.
    chunks = [
        unique_paste_values[i:i + SAP_FILTER_CHUNK_SIZE]
        for i in range(0, len(unique_paste_values), SAP_FILTER_CHUNK_SIZE)
    ]
    if len(chunks) > 1:
        _log(on_event,
             f"splitting {len(unique_paste_values)} key(s) into {len(chunks)} "
             f"chunk(s) of <= {SAP_FILTER_CHUNK_SIZE}")

    # Resolved on the first chunk, after open_ztbv_table has the selection
    # screen up -- a (table, field) pair missing from PUSH_BUTTONS is then
    # found by its on-screen label instead of failing.
    push_id: str | None = None
    frames: list[pd.DataFrame] = []
    ts = int(time.time())
    # Every SAP field this step needs, by technical name -- what the grid
    # reader asks for, and exactly what the original notebook read.
    wanted_cols = list(dict.fromkeys(
        step.sap_key_columns
        + step.sap_output_columns
        + [e.sap_col for e in step.extras if e.sap_col]
    ))
    # One export fallback decision per step, not per chunk: if the grid read
    # is unavailable here it will be unavailable for every other chunk too,
    # and flip-flopping between paths mid-step would mix column namings.
    use_grid = True
    sap_rows_total = 0
    for ci, chunk in enumerate(chunks):
        if stop.is_set():
            raise Cancelled()
        tag = f" (chunk {ci + 1}/{len(chunks)})" if len(chunks) > 1 else ""

        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: loading {step.sap_table}{tag}")
        open_ztbv_table(sap, step.sap_table, log=lambda m: _log(on_event, m))
        if push_id is None:
            push_id = resolve_push_button(
                sap, step.sap_table, step.push_button_field,
                log=lambda m: _log(on_event, m))
        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: sending {len(chunk)} filter values{tag}")
        try:
            # Item 3: primary transport is a temp text file ('Import from
            # Text File' in the multi-select dialog) -- deterministic even
            # if the user copies something to the clipboard mid-run.
            fill_multi_value_filter_from_file(
                sap, push_id, chunk, tmp_dir, log=lambda m: _log(on_event, m))
        except SapError as e:
            # Fallback: the Gen 2/3 clipboard path (values staged via an
            # Excel scratch workbook). Kept for SAP GUI versions where the
            # import dialog is not scriptable.
            _log(on_event,
                 f"file import unavailable ({e}); falling back to "
                 f"clipboard paste", "warn")
            scratch = stage_values_on_clipboard(xl.app, chunk)
            try:
                paste_multi_value_filter(
                    sap, push_id, chunk, log=lambda m: _log(on_event, m))
            finally:
                close_scratch(scratch)
        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: executing query{tag}")
        execute_query(sap, log=lambda m: _log(on_event, m))

        # Item 2: read the status bar + confirm a result grid exists. An SAP
        # error message or missing grid fails HERE with SAP's own words
        # instead of a confusing export failure two calls later.
        grid_rows = query_result_check(sap, log=lambda m: _log(on_event, m))
        sap_rows_total += grid_rows
        if grid_rows == 0:
            _log(on_event, f"SAP returned 0 rows{tag}; nothing to export", "warn")
            sub_progress(2 + 4 * (ci + 1) / len(chunks))
            continue

        # --- Primary read: the grid itself, by technical field name ------
        # Ported from the original notebook. It sidesteps the export file
        # format entirely and, because technical names are unique per field,
        # every alias / duplicate-title / short-label problem with it.
        if use_grid:
            try:
                _status(on_event,
                        f"[{step_index + 1}/{total_steps}] SAP: reading ALV grid "
                        f"({grid_rows} rows){tag}")
                read_start = time.time()
                records = read_alv_grid(
                    sap, wanted_cols, log=lambda m: _log(on_event, m), stop=stop)
                if stop.is_set():
                    raise Cancelled()
                df_chunk = pd.DataFrame(records, columns=wanted_cols)
                _log(on_event,
                     f"grid read finished in {time.time() - read_start:.1f}s "
                     f"({len(df_chunk)} rows)")
                if len(df_chunk) < grid_rows:
                    raise RuntimeError(
                        f"Grid read row-count mismatch{tag}: the status bar "
                        f"reports {grid_rows} row(s) but only "
                        f"{len(df_chunk)} were read.")
                frames.append(df_chunk)
                sub_progress(2 + 4 * (ci + 1) / len(chunks))
                continue
            except SapError as e:
                # A missing technical name is the expected reason to land
                # here; fall back to the export for the rest of the step.
                _log(on_event,
                     f"grid read unavailable ({e}); falling back to the file "
                     f"export for this step", "warn")
                use_grid = False

        _status(on_event,
                f"[{step_index + 1}/{total_steps}] SAP: exporting ALV grid "
                f"({grid_rows} rows){tag}")
        export_name = f"{step.sap_table}_{step.push_button_field}_{ts}_{ci}.xlsx"
        export_start = time.time()
        export_path = export_alv_to_file(
            sap, tmp_dir, export_name, log=lambda m: _log(on_event, m)
        )
        try:
            size_kb = os.path.getsize(export_path) / 1024.0
        except OSError:
            size_kb = 0.0
        _log(on_event, f"export finished in {time.time() - export_start:.1f}s "
                        f"({size_kb:.0f} KB)")

        df_chunk = _read_sap_export(
            export_path, log=lambda m: _log(on_event, m)
        )
        # Item 2: verify the file matches what the grid showed. Fewer rows
        # than the grid = truncated export = silent data loss downstream.
        if len(df_chunk) < grid_rows:
            raise RuntimeError(
                f"Export row-count mismatch{tag}: the SAP grid shows "
                f"{grid_rows} row(s) but the export file contains "
                f"{len(df_chunk)} -- the export is incomplete. Re-run; if it "
                f"persists, export once manually from ZTBV and compare with "
                f"{export_path}.")
        if len(df_chunk) > grid_rows:
            _log(on_event,
                 f"note: export has {len(df_chunk)} row(s) vs {grid_rows} in "
                 f"the grid (totals/subtotal lines?); harmless unless the "
                 f"extra rows carry key values", "warn")
        frames.append(df_chunk)
        sub_progress(2 + 4 * (ci + 1) / len(chunks))

    # --- 3. Combine chunk results, build lookup dict ---------------------
    _status(on_event, f"[{step_index + 1}/{total_steps}] loading SAP export")
    extra_sap_cols = [e.sap_col for e in step.extras if e.sap_col]
    if not frames:
        # Every chunk came back empty: a legitimate "no matches" outcome.
        # (Previously this crashed in _build_lookup with a missing-columns
        # error, because the export of an empty grid carries no data.)
        _log(on_event,
             "SAP returned no rows for any of the requested keys -- every "
             "row will be treated as unmatched", "warn")
        df = None
        lookup, dup_count, blank_key_count = {}, 0, 0
    else:
        df = frames[0] if len(frames) == 1 else pd.concat(frames, ignore_index=True)
        _log(on_event, f"SAP returned {len(df)} row(s) with columns {list(df.columns)[:8]}{'...' if len(df.columns) > 8 else ''}")
        lookup, dup_count, blank_key_count = _build_lookup(
            df,
            step.sap_key_columns,
            step.sap_output_columns + extra_sap_cols,
        )
    if dup_count:
        _log(on_event,
             f"WARNING: SAP returned {dup_count} duplicate key(s); only the "
             f"FIRST row per key is used. Check the run log file (path echoed "
             f"above) for the raw SAP columns, or tighten your ALV filter.",
             "warn")
    if blank_key_count:
        _log(on_event,
             f"note: skipped {blank_key_count} SAP row(s) with blank/partial "
             f"key columns", "info")

    # --- 5. Resolve each row's fate: matched vs. else --------------------
    # Skip rows use the same else branch as SAP non-matches -- the notebook
    # write loop makes no distinction (both fall through to the same clear
    # branch). See _SKIP_KEY_MARKERS docstring.
    _status(on_event, f"[{step_index + 1}/{total_steps}] matching keys in memory")
    matched = 0
    entries_by_row: list[dict | None] = []
    for k, skipped in zip(excel_keys, skip_flags):
        if skipped:
            entries_by_row.append(None)
            continue
        entry = lookup.get(k)
        entries_by_row.append(entry)
        if entry is not None:
            matched += 1

    # --- 6. Publish outputs for downstream steps (Gen 4) -----------------
    # Later steps key off these values from memory instead of re-reading an
    # Excel write-back.
    for j, sap_c in enumerate(step.sap_output_columns):
        col_values = []
        for entry in entries_by_row:
            v = entry[sap_c] if entry is not None else ""
            col_values.append("" if v is None else v)
        vsheet.publish(step.excel_output_columns[j], col_values)
    # Extras are published too when they are a pure function of the fetch
    # (a SAP source and no preserve-on-nonmatch dependence on existing cell
    # content): TO step 2 keys on N/O, which step 1 derives from ABLAD.
    for extra in step.extras:
        if extra.sap_col is None or extra.preserve_on_nonmatch:
            continue
        col_values = []
        for entry in entries_by_row:
            if entry is None:
                col_values.append("")
                continue
            v = entry[extra.sap_col]
            v = "" if v is None else v
            if extra.transform is not None:
                v = extra.transform(v)
            col_values.append(v)
        vsheet.publish(extra.excel_col, col_values)

    non_skip = n_rows - n_skipped
    unmatched = non_skip - matched
    # Sanity: if we sent >20 unique keys and SAP came back with <10% as many
    # rows as we asked about, something is off -- surface a warning so the
    # user does not silently accept mostly-empty output.
    if len(unique_paste_values) > 20 and len(lookup) < max(1, len(unique_paste_values) // 10):
        _log(on_event,
             f"WARNING: sent {len(unique_paste_values)} unique key(s) to SAP but "
             f"only {len(lookup)} matched. Common causes: (a) query returned an "
             f"error screen, (b) the ALV filter rejected the values, (c) plant "
             f"or table wrong for this environment.", "warn")
    _log(on_event,
         f"Step {step_index + 1} fetched in {time.time() - step_started:.1f}s: "
         f"{matched}/{non_skip} rows matched, {unmatched} unmatched"
         + (f", {n_skipped} skipped" if n_skipped else "")
         + " (write deferred to final pass)",
         "ok")
    _progress(on_event, (step_index + 1) / n_segments)
    result = StepResult(
        step=step,
        step_index=step_index,
        last_row=last_row,
        excel_keys=excel_keys,
        entries_by_row=entries_by_row,
        n_skipped=n_skipped,
        matched=matched,
        unmatched=unmatched,
        sap_rows=sap_rows_total,
    )
    if step_popups:
        # The original notebook's per-step message box, counts and all. The
        # operator can look at the SAP result grid (still on screen) before
        # clicking OK; Cancel stops the run with the workbook untouched.
        key_cols = "/".join(_col_letter(c) for c in sorted(step.key_columns))
        out_cols = ", ".join(_step_columns(step))
        _step_popup(
            on_event, stop,
            f"Step {step_index + 1} of {total_steps} completed",
            f"{step.name}\n\n"
            f"SAP {step.sap_table} rows detected: {sap_rows_total}\n"
            f"Matched: {matched}\n"
            f"Not matched: {unmatched}\n"
            f"Skipped (blank / NOT FOUND): {n_skipped}\n\n"
            f"Keys were read from column(s) {key_cols}.\n"
            f"Results go to column(s) {out_cols}, written together at the "
            f"end of the run.\n\n"
            f"OK = continue, Cancel = stop the run (workbook untouched).")
    return result


def _write_back(
    results: list[StepResult],
    xl: ExcelCtx,
    on_event: EventFn,
    total_steps: int,
) -> None:
    """Apply every fetched step's writes to the workbook in ONE pass.

    Gen 4: nothing reaches the workbook unless every step fetched
    successfully -- a SAP failure or user Stop during the fetch phase leaves
    the file completely untouched. Per-step write semantics (clear-to-bottom,
    text formats, extras preserve/clear rules, audit match-key column) are
    unchanged from Gen 2/3; only WHEN they run has moved. Step column sets
    are disjoint in both workflows, so applying them sequentially here is
    identical to the old interleaved order.
    """
    n_segments = total_steps + 1
    _status(on_event, "writing all step results to Excel (single pass)")
    sheet = xl.sheet
    rows_count = int(sheet.Rows.Count)

    with bulk_write(xl.app):
        for done, res in enumerate(results):
            step = res.step
            last_row = res.last_row
            excel_keys = res.excel_keys
            entries_by_row = res.entries_by_row
            n_rows = len(excel_keys)

            # -------- Main output block ---------------------------------
            oc_min = min(step.excel_output_columns)
            oc_max = max(step.excel_output_columns)
            oc_span = oc_max - oc_min + 1
            target_write = _range(oc_min, oc_max, 2, last_row)
            # Match notebook: clear the ENTIRE column range down to
            # Rows.Count so stale rows below last_row (from a prior, longer
            # run) are wiped.
            target_clear = _range(oc_min, oc_max, 2, rows_count)

            main_out: list[list] = []
            for i in range(n_rows):
                entry = entries_by_row[i]
                row_vals = ["" for _ in range(oc_span)]
                if entry is not None:
                    for j, sap_c in enumerate(step.sap_output_columns):
                        excel_c = step.excel_output_columns[j]
                        offset = excel_c - oc_min
                        val = entry[sap_c]
                        row_vals[offset] = "" if val is None else val
                main_out.append(row_vals)

            clear_range(sheet, target_clear)
            set_column_format_text(
                sheet, f"{_col_letter(oc_min)}:{_col_letter(oc_max)}"
            )
            write_range_2d(sheet, target_write, main_out)

            # -------- Extras (per-column, with per-column semantics) ----
            for extra in step.extras:
                letter = _col_letter(extra.excel_col)
                xrange = f"{letter}2:{letter}{last_row}"

                # Read existing only when at least one row will preserve it.
                # Safe to read here (not in the fetch phase): no step writes
                # another step's extras column, so the pre-run content is
                # still intact at this point.
                need_existing = extra.preserve_on_nonmatch or extra.sap_col is None
                existing_extra = (
                    read_range_2d(sheet, xrange) if need_existing else None
                )

                col_data: list[list] = []
                for i in range(n_rows):
                    existing_val = (
                        existing_extra[i][0]
                        if (existing_extra and i < len(existing_extra)
                            and existing_extra[i])
                        else ""
                    )
                    entry = entries_by_row[i]
                    if entry is not None:
                        # Matched row.
                        if extra.sap_col is None:
                            # No SAP source -> preserve on match
                            # (mirrors notebook TO-3 col J behavior).
                            col_data.append([existing_val])
                        else:
                            v = entry[extra.sap_col]
                            v = "" if v is None else v
                            if extra.transform is not None:
                                v = extra.transform(v)
                            col_data.append([v])
                    else:
                        # Non-match OR skip (treated the same, per notebook).
                        if extra.preserve_on_nonmatch:
                            col_data.append([existing_val])
                        else:
                            col_data.append([""])

                # Only set text format on columns where we're writing SAP
                # data; a preserve-only column (sap_col=None) should keep
                # whatever NumberFormat the user had -- notebook doesn't
                # touch it.
                if extra.sap_col is not None:
                    set_column_format_text(sheet, f"{letter}:{letter}")
                write_range_2d(sheet, xrange, col_data)

            # -------- Match key column (P for TO-2) ----------------------
            if step.match_key_column is not None:
                mk_letter = _col_letter(step.match_key_column)
                # Header at row 1 + text format for the whole column,
                # matching notebook lines 851-852.
                sheet.Cells(1, step.match_key_column).Value = step.match_key_header
                set_column_format_text(sheet, f"{mk_letter}:{mk_letter}")
                # Notebook line 863 writes the composite key to P for every
                # row in the loop (including skip rows, which get
                # "NOTFOUND|..." or "|" written). Match that by writing
                # excel_keys[i] for every row 2..last_row.
                mk_range = f"{mk_letter}2:{mk_letter}{last_row}"
                mk_data = [[k] for k in excel_keys]
                write_range_2d(sheet, mk_range, mk_data)

            _progress(
                on_event,
                (total_steps + (done + 1) / max(1, len(results))) / n_segments,
            )

    _log(on_event,
         f"write-back complete: {len(results)} step block(s) written in one pass",
         "ok")


def _step_columns(step: LookupStep) -> list[str]:
    """Every Excel column a step writes, as letters, in write order."""
    cols = [_col_letter(c) for c in step.excel_output_columns]
    cols += [_col_letter(e.excel_col) for e in step.extras]
    if step.match_key_column is not None:
        cols.append(_col_letter(step.match_key_column))
    return cols


def _salvage_completed_steps(
    results: list[StepResult],
    steps: list[LookupStep],
    xl,
    on_event: EventFn,
    write_state: str,
    dry_run: bool,
) -> str:
    """A step failed. Write the steps that DID complete, and say plainly
    which columns are filled and which are not.

    Gen 4 originally discarded everything on any failure. That is the safest
    rule but it also threw away work that was already correct -- a run whose
    step 1 matched every row still left the workbook untouched because step 2
    could not find its SAP filter. Steps write disjoint column blocks and
    `results` only ever holds a PREFIX of the workflow (a step that fails
    stops the ones after it, which is also what feeds them their keys), so
    applying that prefix is exactly what a successful run would have written
    for those steps.

    Returns (new_write_state, salvage_error_text). Never raises: a failure
    to salvage must not replace the original error -- but it must not be
    silent either, so its text travels back for the error popup (two
    failures in one run usually share one cause).
    """
    if dry_run or not results or xl is None:
        return write_state, ""
    if write_state != "none":
        # The failure was already inside the final write-back; re-running it
        # would double-apply. Leave it to the 'partial' warning.
        return write_state, ""

    done = {res.step_index for res in results}
    missing = [(i, s) for i, s in enumerate(steps) if i not in done]
    try:
        _log(on_event,
             f"salvage: {len(results)} of {len(steps)} step(s) completed "
             f"before the failure -- writing those, leaving the rest alone",
             "warn")
        _write_back(results, xl, on_event, len(steps))
        try:
            excel_save(xl.book)
        except ExcelError as e:
            _log(on_event, f"WARNING: {e}", "warn")
        for res in results:
            _log(on_event,
                 f"  WRITTEN  step {res.step_index + 1} "
                 f"({res.step.sap_table}): columns "
                 f"{', '.join(_step_columns(res.step))} -- "
                 f"{res.matched} matched / {res.unmatched} unmatched", "ok")
        for i, s in missing:
            _log(on_event,
                 f"  NOT RUN  step {i + 1} ({s.sap_table}): columns "
                 f"{', '.join(_step_columns(s))} left as they were", "warn")
        return "salvaged", ""
    except Exception as e:
        _log(on_event,
             f"WARNING: could not write the completed steps either ({e}). "
             f"The workbook may be partially updated -- review it before "
             f"saving.", "warn")
        _file_log_traceback()
        return "partial", f"{type(e).__name__}: {e}"


def _write_state_text(write_state: str) -> str:
    """One plain sentence about the workbook, shared by the log lines and
    the popups so the two can never disagree."""
    return {
        "none": "The workbook was NOT modified.",
        "salvaged": ("The workbook holds the completed steps ONLY (see "
                     "WRITTEN / NOT RUN in the log) and has been saved. "
                     "Re-running after the fix is safe -- every step "
                     "rewrites its own columns from scratch."),
        "partial": ("Failure happened DURING the final write-back -- the "
                    "workbook may be partially updated and has NOT been "
                    "saved. Review it before saving manually."),
        "done": ("All output columns were written in one pass and the "
                 "workbook was SAVED."),
    }[write_state]


def _run_summary_lines(results: list[StepResult]) -> str:
    """Per-step counts for the completion popup, one line per step ran."""
    if not results:
        return "No step had anything to look up."
    return "\n".join(
        f"Step {r.step_index + 1} ({r.step.sap_table}): "
        f"{r.matched} matched / {r.unmatched} not matched"
        + (f" / {r.n_skipped} skipped" if r.n_skipped else "")
        + f" -> columns {', '.join(_step_columns(r.step))}"
        for r in results)


def _log_write_state(on_event: EventFn, write_state: str) -> None:
    """After a cancel/failure, tell the user exactly what state the file is
    in."""
    if write_state == "none":
        _log(on_event,
             _write_state_text("none") + " (no step completed before the "
             "failure)", "info")
    elif write_state == "salvaged":
        _log(on_event, _write_state_text("salvaged"), "warn")
    elif write_state == "partial":
        _log(on_event, "WARNING: " + _write_state_text("partial"), "warn")


def _dry_run_report(results: list[StepResult], on_event: EventFn) -> None:
    """Item 4: summarize what the write-back WOULD do, without doing it."""
    _log(on_event, "DRY RUN -- the workbook will not be modified. Planned writes:")
    for res in results:
        step = res.step
        cols = _step_columns(step)
        _log(on_event,
             f"  step {res.step_index + 1} ({step.sap_table}): "
             f"{res.matched} matched / {res.unmatched} unmatched"
             + (f" / {res.n_skipped} skipped" if res.n_skipped else "")
             + f" -> columns {'/'.join(cols)}, rows 2..{res.last_row}")
        shown = 0
        for i, entry in enumerate(res.entries_by_row):
            if entry is None:
                continue
            preview = ", ".join(f"{k}={entry[k]}" for k in entry)
            _log(on_event, f"    e.g. row {i + 2}: {preview}")
            shown += 1
            if shown >= 3:
                break


def _diagnose(workflow: str, sap: SapSession, on_event: EventFn) -> None:
    """Dump every ZTBV selection screen this workflow touches.

    ZTBV's select-options are named generically (S3, S15, S29), so a filter
    can only be identified by the label printed beside it. When that pairing
    fails -- a screen built without GuiLabels, an unusual SAP GUI build --
    the run dies with nothing to act on. This walks each table's screen and
    logs BOTH the resolved filter list and the raw control inventory, so the
    S<n> -> field mapping can be read by eye from the log file and pinned
    with ESA_LOOKUP_PUSH_<TABLE>_<FIELD>.

    Read-only: it navigates and reads. No filter is pasted, no query is
    executed, and Excel is never opened.
    """
    tables: list[str] = []
    for step in WORKFLOWS[workflow]:
        if step.sap_table not in tables:
            tables.append(step.sap_table)

    _log(on_event,
         f"DIAGNOSE: dumping the ZTBV selection screen for {len(tables)} "
         f"table(s): {', '.join(tables)}. Nothing is pasted, executed, or "
         f"written -- this only reads the screens.")

    wanted_by_table: dict[str, list[str]] = {}
    for step in WORKFLOWS[workflow]:
        wanted_by_table.setdefault(step.sap_table, [])
        if step.push_button_field not in wanted_by_table[step.sap_table]:
            wanted_by_table[step.sap_table].append(step.push_button_field)

    for table in tables:
        _log(on_event, "")
        _log(on_event, f"===== {table} =====")
        open_ztbv_table(sap, table, log=lambda m: _log(on_event, m))
        try:
            filters = describe_selection_screen(sap)
        except SapError as e:
            _log(on_event, f"{table}: cannot read the selection screen: {e}",
                 "error")
            continue

        _log(on_event, f"{table}: {len(filters)} multi-value filter(s)")
        for f in filters:
            _log(on_event,
                 f"  {f['param']:<6} row={f['row']:<4} label={f['label']!r}"
                 + (f"  tooltip={f['tooltip']!r}" if f["tooltip"] else ""))

        # What this workflow actually needs off this screen, and whether it
        # would resolve right now.
        for field in wanted_by_table[table]:
            try:
                pid = resolve_push_button(sap, table, field)
                _log(on_event, f"  -> {field} resolves to {pid}", "ok")
            except SapError as e:
                _log(on_event, f"  -> {field} DOES NOT RESOLVE: {e}", "error")
                _log(on_event,
                     f"     pin it with: set "
                     f"{env_override_var(table, field)}=S<n>", "warn")

        # Raw inventory -- the fallback when no label paired. Only text-bearing
        # controls; the rest is noise for this purpose.
        _file_log("info", f"--- {table}: raw control inventory ---")
        try:
            for it in screen_inventory(sap):
                if not (it["text"] or it["tooltip"]):
                    continue
                _file_log("info",
                          f"  {it['type']:<16} row={it['row']:<4} "
                          f"col={it['col']:<4} name={it['name']!r} "
                          f"text={it['text']!r} tooltip={it['tooltip']!r}")
        except SapError as e:
            _file_log("info", f"  inventory unavailable: {e}")

    _log(on_event, "")
    _log(on_event,
         "DIAGNOSE complete. The full per-control dump is in the log FILE "
         "(path echoed at the top) -- send that file on.", "ok")


@dataclass
class RunConfig:
    excel_path: str
    workflow: str            # "TO" or "NOTIF"
    stop_event: threading.Event
    dry_run: bool = False    # Item 4: fetch + report matches, write nothing
    diagnose: bool = False   # dump SAP selection screens; touch nothing else
    # Original-notebook rhythm: a blocking counts popup after every step
    # (OK = continue, Cancel = stop). Error and completion popups are always
    # emitted regardless of this flag; this only controls the per-step ones.
    step_popups: bool = False


def run(cfg: RunConfig, on_event: EventFn) -> bool:
    """Entry point invoked on a background thread. Returns True on success."""
    # Fix H: initialize this worker thread's COM apartment. pywin32 does an
    # implicit CoInitialize on Dispatch, but the second Run click (new
    # worker thread, same process) can hit CO_E_NOTINITIALIZED on
    # GetActiveObject / GetObject("SAPGUI") without an explicit init here.
    # Uninitialize in finally so the thread exits clean.
    pythoncom.CoInitialize()
    run_started = time.time()
    log_path = _open_run_log()
    # Gen 4 write state, used by the error/cancel handlers to tell the user
    # exactly what happened to the file: "none" -> nothing written (fetch
    # phase), "partial" -> failure mid write-back, "done" -> fully written.
    write_state = "none"
    # Declared up here so the failure handlers can salvage whatever the fetch
    # phase managed to complete before it died, and so the error popup can
    # name the step and snapshot the SAP screen.
    xl = None
    sap = None
    steps: list[LookupStep] = []
    results: list[StepResult] = []
    step_label = "startup (before any SAP step)"
    salvage_error = ""

    # Remember the last status line: when a run dies, "what was it DOING"
    # is the single most valuable fact on the error popup -- a com_error
    # during 'reading keys' points at Excel, during 'sending filter values'
    # at SAP, without waiting for the log file to travel.
    last_action = {"text": ""}
    _caller_on_event = on_event

    def on_event(kind, payload):  # noqa: shadows the parameter on purpose
        if kind == "status":
            last_action["text"] = payload
        _caller_on_event(kind, payload)

    def _error_popup(err_text: str) -> None:
        """The original notebook's 'Error occurred' message box, upgraded
        with what a remote debugger needs: the step, the exact action in
        flight, the SAP screen at the moment of failure, the workbook
        state, and the log path."""
        snapshot = ""
        if sap is not None:
            try:
                snapshot = screen_snapshot(sap)
            except Exception:
                snapshot = ""
        _popup(
            on_event, "error", "esa-lookup -- error",
            f"Error occurred in {step_label}:\n\n{err_text}\n\n"
            + (f"While doing: {last_action['text']}\n\n"
               if last_action["text"] else "")
            + (f"SAP screen at the moment of failure:\n{snapshot}\n\n"
               if snapshot else "")
            + "The SAP window has been left exactly where it stopped.\n"
              "Please screenshot BOTH the SAP window and this message.\n\n"
            + _write_state_text(write_state)
            + (f"\n\nALSO: writing the completed steps to Excel failed "
               f"too:\n{salvage_error}\nTwo failures in one run usually "
               f"share one cause -- most often Excel or the workbook was "
               f"closed, or was being clicked/edited, while the run was in "
               f"flight. Close the workbook WITHOUT saving and re-run."
               if salvage_error else "")
            + (f"\n\nLog file (attach it when reporting):\n{log_path}"
               if log_path else ""))

    try:
        _log(on_event, f"esa-lookup starting workflow '{cfg.workflow}'", "info")
        if log_path:
            _log(on_event, f"detailed log file: {log_path}", "info")
        # Diagnostic: freeze the environment into the file so a post-mortem
        # can tell which Python / pandas / openpyxl the run used.
        try:
            import openpyxl as _openpyxl
            openpyxl_v = _openpyxl.__version__
        except Exception:
            openpyxl_v = "?"
        _file_log("info",
                  f"env: Python {sys.version.split()[0]} on "
                  f"{sys.platform}, pandas {pd.__version__}, openpyxl {openpyxl_v}, "
                  f"cwd={os.getcwd()}, excel={cfg.excel_path}")

        # Diagnose is SAP-only and read-only: no workbook is opened, so it
        # runs even when the operator has no file picked.
        if cfg.diagnose:
            _status(on_event, "attaching to SAP GUI")
            sap = sap_attach()
            _log(on_event, "attached to SAP GUI session", "ok")
            _diagnose(cfg.workflow, sap, on_event)
            _status(on_event, "diagnose complete -- nothing written")
            _popup(on_event, "info", "Diagnose complete",
                   "The ZTBV selection screens were read and dumped to the "
                   "log. Nothing was pasted, executed, or written.\n\n"
                   + (f"Send this log file:\n{log_path}" if log_path else
                      "See the log pane for the results."))
            _progress(on_event, 1.0)
            on_event("done", True)
            return True

        _status(on_event, "opening Excel")
        xl = excel_attach(cfg.excel_path)
        _log(on_event, f"attached to Excel: {os.path.basename(cfg.excel_path)}", "ok")

        _status(on_event, "attaching to SAP GUI")
        sap = sap_attach()
        _log(on_event, "attached to SAP GUI session", "ok")

        steps = WORKFLOWS[cfg.workflow]
        total_steps = len(steps)
        tmp_dir = os.path.join(tempfile.gettempdir(), "esa_lookup")

        # Pre-flight: verify the workflow's primary input column has data
        # before we even spin up SAP navigation. Later steps derive their
        # own last_row (Fix F) and skip themselves if their column is empty.
        primary_col = steps[0].key_columns[0]
        primary_last = last_row_in_column(xl.sheet, primary_col)
        if primary_last < 2:
            _log(on_event,
                 f"No data below the header in column "
                 f"{_col_letter(primary_col)} (#{primary_col}) -- the "
                 f"workflow's primary input. Aborting.", "error")
            # Notebook parity: "No TO Number found in Column K." was a
            # message box, not a log line.
            _popup(on_event, "error", "esa-lookup -- nothing to do",
                   f"No data found below the header in column "
                   f"{_col_letter(primary_col)} -- the {cfg.workflow} "
                   f"workflow's primary input.\n\nNothing was written.")
            on_event("done", False)
            return False
        _log(on_event,
             f"step 1 will process rows 2..{primary_last} (column "
             f"{_col_letter(primary_col)}); each subsequent step derives "
             f"its own row range from its own key column")
        _log(on_event,
             "Gen 4: steps chain in memory; the workbook is written once, "
             "after every SAP step has succeeded")

        # ---- Phase 1: fetch every step from SAP (no workbook writes) ----
        vsheet = VirtualSheet(xl.sheet)
        totals_matched = 0
        totals_seen = 0
        for i, step in enumerate(steps):
            if cfg.stop_event.is_set():
                raise Cancelled()
            step_label = f"Step {i + 1} of {total_steps} ({step.sap_table})"
            res = _fetch_step(
                step, vsheet, xl, sap, tmp_dir, on_event, i, total_steps,
                cfg.stop_event, step_popups=cfg.step_popups,
            )
            if res is None:
                continue  # no-op step (empty key column)
            results.append(res)
            totals_matched += res.matched
            totals_seen += res.matched + res.unmatched

        if cfg.stop_event.is_set():
            raise Cancelled()

        # ---- Dry run: report what would be written, touch nothing -------
        if cfg.dry_run:
            _dry_run_report(results, on_event)
            _status(on_event, "dry run complete -- nothing written")
            _log(on_event,
                 f"DRY RUN complete: {totals_matched}/{totals_seen} "
                 f"row-matches across {total_steps} step(s); the workbook "
                 f"was not modified", "ok")
            _popup(on_event, "info", "Dry run complete",
                   _run_summary_lines(results)
                   + "\n\nDRY RUN -- the workbook was NOT modified.")
            _progress(on_event, 1.0)
            on_event("done", True)
            return True

        # ---- Phase 2: single write-back pass + save ---------------------
        step_label = "the final write-back to Excel"
        write_state = "partial"
        _write_back(results, xl, on_event, total_steps)
        write_state = "done"

        # Fix L: save() now raises ExcelError on failure; warn the user
        # rather than silently pretending the write persisted.
        try:
            excel_save(xl.book)
        except ExcelError as e:
            _log(on_event, f"WARNING: {e}", "warn")

        _status(on_event, "done")
        _log(on_event, f"all steps complete: {totals_matched}/{totals_seen} row-matches across {total_steps} step(s)", "ok")
        # Notebook parity: the "Completed." message box with the counts.
        _popup(on_event, "info", "Completed",
               _run_summary_lines(results) + "\n\n"
               + _write_state_text("done"))
        _progress(on_event, 1.0)
        on_event("done", True)
        return True

    except Cancelled:
        # Stop stays strictly all-or-nothing: the GUI promises the file will
        # not be modified, and an explicit abort is not a partial result the
        # operator asked to keep.
        _log(on_event, "cancelled by user", "warn")
        if write_state == "none":
            _log(on_event,
                 "the workbook was NOT modified (Stop leaves the file "
                 "untouched)", "info")
        else:
            _log_write_state(on_event, write_state)
        _popup(on_event, "info", "Cancelled",
               "The run was stopped.\n\n" + _write_state_text(write_state))
        _status(on_event, "cancelled")
        on_event("done", False)
        return False
    except SapError as e:
        _log(on_event, f"SAP error: {e}", "error")
        _file_log_traceback()  # full chain into the log file for post-mortem
        write_state, salvage_error = _salvage_completed_steps(
            results, steps, xl, on_event, write_state, cfg.dry_run)
        _log_write_state(on_event, write_state)
        _error_popup(str(e))
        _status(on_event, "SAP error")
        on_event("done", False)
        return False
    except ExcelError as e:
        _log(on_event, f"Excel error: {e}", "error")
        _file_log_traceback()
        write_state, salvage_error = _salvage_completed_steps(
            results, steps, xl, on_event, write_state, cfg.dry_run)
        _log_write_state(on_event, write_state)
        _error_popup(str(e))
        _status(on_event, "Excel error")
        on_event("done", False)
        return False
    except Exception as e:
        _log(on_event, f"unexpected error: {e}", "error")
        _log(on_event, traceback.format_exc(), "error")
        _file_log_traceback()
        write_state, salvage_error = _salvage_completed_steps(
            results, steps, xl, on_event, write_state, cfg.dry_run)
        _log_write_state(on_event, write_state)
        _error_popup(f"{type(e).__name__}: {e}")
        _status(on_event, "failed")
        on_event("done", False)
        return False
    finally:
        _log(on_event, f"total elapsed: {time.time() - run_started:.1f}s", "info")
        _close_run_log()
        try:
            pythoncom.CoUninitialize()
        except Exception:
            pass


# ---------------------------------------------------------------------------
# Notebook front end: message boxes + run_process(), the ONLY function the
# process cells call. Everything above this line is shared machinery.
# ---------------------------------------------------------------------------

_MB_OKCANCEL, _MB_ICONERROR, _MB_ICONINFO, _MB_TOPMOST = 0x1, 0x10, 0x40, 0x40000


def _msgbox(title: str, message: str, flags: int) -> int:
    """The original notebook's ctypes message box. Returns the button id
    (1 = OK, 2 = Cancel)."""
    import ctypes
    return ctypes.windll.user32.MessageBoxW(0, message, title, flags)


def make_event_handler(popups: bool):
    """Log lines print into the cell output; popup events become real
    Windows message boxes when popups=True, printed banners when False.
    ERROR popups always box regardless -- they only appear when something is
    wrong, which is exactly when the box is wanted."""
    boxes_possible = sys.platform == "win32"

    def on_event(kind, payload):
        if kind == "log":
            msg, level = payload
            prefix = {"ok": "[OK]  ", "warn": "[WARN]", "error": "[ERR] ",
                      "info": "      "}.get(level, "      ")
            print(f"{prefix} {msg}")
        elif kind == "status":
            print(f"...    {payload}")
        elif kind == "popup":
            p = payload
            banner = "!" if p.get("kind") == "error" else "-"
            print(banner * 60)
            print(f"[{p.get('kind', 'info').upper()}] {p['title']}")
            print(p["message"])
            print(banner * 60)
            if p.get("ack") is not None:                     # step popup
                proceed = True
                if popups and boxes_possible:
                    ret = _msgbox(p["title"], p["message"],
                                  _MB_OKCANCEL | _MB_ICONINFO | _MB_TOPMOST)
                    proceed = (ret == 1)
                if p.get("result") is not None:
                    p["result"]["proceed"] = proceed
                p["ack"].set()
            elif p.get("kind") == "error":
                if boxes_possible:
                    _msgbox(p["title"], p["message"],
                            _MB_ICONERROR | _MB_TOPMOST)
            elif popups and boxes_possible:
                _msgbox(p["title"], p["message"],
                        _MB_ICONINFO | _MB_TOPMOST)
        elif kind == "done":
            ok = payload
            print(f"===== process {'succeeded' if ok else 'FAILED'} =====")

    return on_event


# Remembered across cells, so Run All asks for the workbook ONCE and the
# Notification cell re-uses the file the TO cell picked.
_last_browsed = ""


def run_process(workflow: str, excel_path: str = "", popups: bool = True,
                dry_run: bool = False, diagnose: bool = False) -> bool:
    """Run one process end to end.

    workflow    "TO" or "NOTIF"
    excel_path  full path to the workbook; "" opens a Browse dialog the
                first time and re-uses that file for later cells in the
                same kernel session (restart the kernel, or set EXCEL_PATH,
                to pick a different file)
    popups      message box after each step (OK = continue, Cancel = stop).
                One-line off switch in the process cell. Error boxes always
                show regardless.
    dry_run     look everything up, report counts, write nothing
    diagnose    dump the ZTBV selection screens instead of processing
                (no Excel file needed)

    On failure or Cancel this raises SystemExit, so a Run All stops HERE
    instead of blindly executing the next process cell.
    """
    global _last_browsed
    if workflow not in WORKFLOWS:
        raise SystemExit(
            f"workflow must be one of {list(WORKFLOWS)}, got {workflow!r}")
    if not diagnose and not excel_path:
        if _last_browsed:
            excel_path = _last_browsed
            print(f"Re-using the workbook selected earlier this session:")
            print(f"  {excel_path}")
            print("(set EXCEL_PATH in the cell, or restart the kernel, to "
                  "pick a different file)")
        else:
            import tkinter as tk
            from tkinter import filedialog
            _root = tk.Tk()
            _root.withdraw()
            _root.attributes("-topmost", True)
            try:
                excel_path = filedialog.askopenfilename(
                    title="Select the Excel workbook to process",
                    filetypes=[("Excel workbooks", "*.xlsx *.xlsm *.xlsb"),
                               ("All files", "*.*")],
                )
            finally:
                _root.destroy()
            if not excel_path:
                raise SystemExit(
                    "No file selected -- nothing was run, and the cells "
                    "after this one were stopped.")
            _last_browsed = excel_path
    cfg = RunConfig(
        excel_path=excel_path,
        workflow=workflow,
        stop_event=threading.Event(),
        dry_run=dry_run,
        diagnose=diagnose,
        step_popups=popups,
    )
    ok = run(cfg, on_event=make_event_handler(popups))
    if not ok:
        # Stop a Run All at the point of failure. The log above (and the
        # log file) say what happened; the next process must not run on top
        # of a failed or cancelled one.
        raise SystemExit(
            "This process did not complete (failed or cancelled) -- the "
            "cells after this one were stopped. Fix the issue and re-run "
            "this cell.")
    return ok


print("Utility class for data pulling: loaded. Now run your process cell below")
print("(TO Number Process or Notification Number Process).")


# TO Number Process

| Step | Reads | SAP table | Writes |
|------|-------|-----------|--------|
| 1 | Col **K** (TO Number) | `LTAP` | Unloading Point -> **M**, and the reservation pair hidden inside it -> **N + O** |
| 2 | Col **N + O** (filled by step 1) | `Z50CFG_ENG_CRNT` | Notification -> **A** (matched rows only), Object / Material / Qty -> **C, D, E**, match key -> **P** |
| 3 | Col **C** (Object Number) | `Z50CFG_ENG_VALD` | Section / Module / Description / Sales Doc. -> **F, G, H, I** |

Step 1 splits the unloading point (`05175455500001`) into the
reservation number (`517545550`) and item (`1`) itself -- the Excel
template's old formula in N/O is no longer needed. Rows without a TO
number or unloading point are skipped, and their column A is left
untouched so the Notification process can fill them next.


In [ ]:
# =========================================================
# TO Number Process   (run the "Utility class for data pulling" cell first)
# =========================================================
POPUPS = True     # message box after each step (OK = continue, Cancel =
                  # stop). ONE-LINE OFF SWITCH: set to False once the first
                  # days have gone smoothly. Error boxes always show.
DRY_RUN = False   # True = report match counts only, write nothing
EXCEL_PATH = r""  # paste the workbook's full path here, or leave "" to browse

run_process("TO", excel_path=EXCEL_PATH, popups=POPUPS, dry_run=DRY_RUN)


# Notification Number Process

| Step | Reads | SAP table | Writes |
|------|-------|-----------|--------|
| 1 | Col **A** (Notification) | `Z50CFG_ENG_CRNT` | Object / Material / Qty -> **C, D, E** |
| 2 | Col **C** (Object Number) | `Z50CFG_ENG_VALD` | Section / Module / Description / Sales Doc. / LID -> **F..J** |

On a combined sheet, run this **after** the TO process -- this is the pass
that fills the notification-only rows.


In [ ]:
# =========================================================
# Notification Number Process   (run the "Utility class" cell first)
# =========================================================
POPUPS = True     # message box after each step -- same one-line switch
DRY_RUN = False   # True = report match counts only, write nothing
EXCEL_PATH = r""  # paste the workbook's full path here, or leave "" to browse

run_process("NOTIF", excel_path=EXCEL_PATH, popups=POPUPS, dry_run=DRY_RUN)


## Troubleshooting

- **A step looks wrong**: press **Cancel** on its message box -- the run
  stops and the workbook is untouched. Then re-run with `DRY_RUN = True`
  to inspect match counts without writing.
- **"Could not resolve the ... filter"**: run the Diagnose cell below and
  send the log file it names.
- Every popup's text is also in the log file, so nothing is lost when a
  box is dismissed. Logs: `%LOCALAPPDATA%\esa-lookup\logs\`.


In [ ]:
# Only needed when a run stops with "Could not resolve the ... filter".
# Reads the ZTBV selection screens for the chosen process and prints which
# S<n> filter slot is which. No Excel file, no query, nothing written.
# Send the log file it names.
#
# Deliberately COMMENTED OUT so a Run All never triggers it -- uncomment
# the next line only when you need it, then run this cell:
# run_process("TO", diagnose=True)          # or "NOTIF"

# If Diagnose names the right slot, pin it here and re-run your process --
# example (uncomment and adjust):
# PUSH_BUTTONS[("Z50CFG_ENG_CRNT", "RSNUM")] = push_button_id("S15")
print("Diagnose cell: nothing to do (see the comments in this cell).")
